# 实验 2：解剖 Qwen3-0.6B 的一次完整前向传播

## 今天回答的问题

**Qwen3-0.6B 对一段真实 token 序列究竟进行了哪些计算，这些计算能否被我们自己重新实现并验证？**

实验 1 画出了模型的外部骨架，但没有任何数据真的流过去。本实验补上这一步：让一个真实句子从 `input_ids` 出发，
沿真实 forward 路径走完 28 个 Decoder Layer，直到 logits 和 Top-K 预测。

本实验采用**两遍结构**：

- **第一遍：观察。** 运行官方 `Qwen3ForCausalLM`，用 hooks 记录每个关键节点的中间状态。
- **第二遍：复现。** 用 PyTorch 基础算子重写每一个模块，参数全部取自真实模型，**不重新初始化**，
  然后与第一遍的记录逐节点对齐。

### 本实验不做的事

- 不研究 tokenizer 如何切词。它只作为入口工具：`自然语言 → tokenizer → input_ids`，从 `input_ids` 开始解剖。
- 不讨论"Qwen3 为什么这样设计"，只弄清"它究竟怎么算"。
- 不使用 KV Cache，只研究完整序列的一次 forward（prefill）。
- 不涉及训练、梯度、采样随机性。

### 真实性原则

不凭记忆猜测计算过程。事实来源按优先级：**本地模型配置 → 本地模型权重 → 实际 Transformers 源码 →
官方模型实际运行结果**。每个重要模块都会记录类名、函数名和源码行号，形成证据链。

## 0. 环境与实验对象核对

正式开始前先确认：我们解剖的是哪一个模型文件，用的是哪一份源码。
后续所有"源码位置"都指向下面这个 `modeling_qwen3.py` 的实际路径。

In [1]:
import inspect
import json
import unicodedata
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.qwen3 import modeling_qwen3 as Q3


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'
SOURCE_PATH = Path(Q3.__file__)

print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'模型路径:      {MODEL_PATH}')
print(f'源码路径:      {SOURCE_PATH}')
print(f'源码相对位置:  .../{SOURCE_PATH.relative_to(SOURCE_PATH.parents[4])}')

torch:        2.13.0+cu130
transformers: 5.15.1
模型路径:      /home/linjunjie/Workspace/xxdw1/models/Qwen3-0.6B-Base
源码路径:      /home/linjunjie/Workspace/xxdw1/.venv/lib/python3.14/site-packages/transformers/models/qwen3/modeling_qwen3.py
源码相对位置:  .../site-packages/transformers/models/qwen3/modeling_qwen3.py


In [2]:
# 先读磁盘上的 config.json 原文，再读 transformers 解析后的 config 对象。
# 两者不完全一致，这个差异本身就是"以实际运行源码为准"的一个实例。
raw_config = json.loads((MODEL_PATH / 'config.json').read_text())

for key in ['model_type', 'hidden_size', 'num_hidden_layers', 'num_attention_heads',
            'num_key_value_heads', 'head_dim', 'intermediate_size', 'vocab_size',
            'rms_norm_eps', 'rope_theta', 'tie_word_embeddings', 'torch_dtype',
            'attention_bias', 'hidden_act', 'sliding_window']:
    print(f'{key:22s} {raw_config.get(key)}')

model_type             qwen3
hidden_size            1024
num_hidden_layers      28
num_attention_heads    16
num_key_value_heads    8
head_dim               128
intermediate_size      3072
vocab_size             151936
rms_norm_eps           1e-06
rope_theta             1000000
tie_word_embeddings    True
torch_dtype            bfloat16
attention_bias         False
hidden_act             silu
sliding_window         None


### 0.1 关于 dtype 的决定

权重在磁盘上是 `bfloat16`。但本实验**统一加载为 `float32`**，理由来自源码本身：

- `Qwen3RMSNorm.forward` 内部强制 `.to(torch.float32)` 再算方差；
- `eager_attention_forward` 里 softmax 指定 `dtype=torch.float32`；
- `Qwen3RotaryEmbedding.forward` 用 `maybe_autocast(enabled=False)` 强制 float32 算 cos/sin。

也就是说，关键数值路径本来就在 float32 上。统一到 float32 后，官方与我们的复现可以对到
相对误差 1e-6 量级；若留在 bf16，误差会放大到 1e-2 量级，逐节点验证就失去意义了。

同时指定 `attn_implementation='eager'`，原因是默认的 sdpa 路径**不返回 attention weights**，
且 causal mask 会是 `None`（靠 `is_causal` 标志隐式处理），我们就没有 mask 张量可观察。

In [3]:
DTYPE = torch.float32
DEVICE = 'cpu'

# 全程关闭梯度。本实验只研究推理，不涉及训练。
# 注意这里用 set_grad_enabled 而不是 torch.inference_mode()：
# inference_mode 产生的张量不能再参与后续（被 autograd 追踪的）计算，
# 而我们要用第一遍抓到的张量喂给第二遍的复现代码。
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=DTYPE,
    attn_implementation='eager',
).to(DEVICE).eval()

config = model.config
print(f'模型类:        {type(model).__name__}')
print(f'device:       {next(model.parameters()).device}')
print(f'dtype:        {next(model.parameters()).dtype}')
print(f'attn 实现:     {config._attn_implementation}')
print(f'training 模式: {model.training}   (应为 False)')
print(f'梯度开启:      {torch.is_grad_enabled()}   (应为 False)')
print(f'参数量:        {sum(p.numel() for p in model.parameters()):,}')

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型类:        Qwen3ForCausalLM
device:       cpu
dtype:        torch.float32
attn 实现:     eager
training 模式: False   (应为 False)
梯度开启:      False   (应为 False)
参数量:        596,049,920


### 0.2 config.json 与 config 对象的一处差异

`config.json` 里 `rope_theta` 是顶层字段，但当前版本的 transformers 把它收进了
`config.rope_parameters` 字典。直接访问 `config.rope_theta` 取不到值。

这类差异正是为什么我们不能凭记忆写代码：下面 RoPE 的实现必须从 `config.rope_parameters` 取
`rope_theta`，才和真实运行的源码一致。

In [4]:
print(f'config.rope_parameters       = {config.rope_parameters}')
print(f"hasattr(config, 'rope_theta') = {hasattr(config, 'rope_theta')}")
print(f'config.layer_types 唯一值     = {set(config.layer_types)}')
print(f'model.model.has_sliding_layers = {model.model.has_sliding_layers}')

config.rope_parameters       = {'rope_theta': 1000000, 'rope_type': 'default'}
hasattr(config, 'rope_theta') = False
config.layer_types 唯一值     = {'full_attention'}
model.model.has_sliding_layers = False


In [5]:
# 后续反复使用的维度常量，全部从 config 读取，不写死数字。
N_LAYERS = config.num_hidden_layers
HIDDEN = config.hidden_size
N_HEADS = config.num_attention_heads
N_KV_HEADS = config.num_key_value_heads
HEAD_DIM = config.head_dim
N_REP = N_HEADS // N_KV_HEADS
INTERMEDIATE = config.intermediate_size
VOCAB = config.vocab_size
RMS_EPS = config.rms_norm_eps
ROPE_THETA = config.rope_parameters['rope_theta']
SCALING = HEAD_DIM ** -0.5

CONSTANTS = [
    ('N_LAYERS',     N_LAYERS,     'Transformer Block 层数'),
    ('HIDDEN',       HIDDEN,       '残差流的宽度，每层进出都是这个数'),
    ('N_HEADS',      N_HEADS,      'query 头数'),
    ('N_KV_HEADS',   N_KV_HEADS,   'key/value 头数'),
    ('N_REP',        N_REP,        '每个 kv 头被几个 q 头共享'),
    ('HEAD_DIM',     HEAD_DIM,     '单头宽度，config 写死，不是 HIDDEN / N_HEADS'),
    ('INTERMEDIATE', INTERMEDIATE, 'MLP 中间层宽度'),
    ('VOCAB',        VOCAB,        '词表大小'),
    ('RMS_EPS',      RMS_EPS,      'RMSNorm 的 eps'),
    ('ROPE_THETA',   ROPE_THETA,   'RoPE 频率基数'),
    ('SCALING',      SCALING,      'attention 缩放系数 = HEAD_DIM ** -0.5'),
]

for name, value, why in CONSTANTS:
    print(f'{name:<13s} {value!s:<20s} {why}')

N_LAYERS      28                   Transformer Block 层数
HIDDEN        1024                 残差流的宽度，每层进出都是这个数
N_HEADS       16                   query 头数
N_KV_HEADS    8                    key/value 头数
N_REP         2                    每个 kv 头被几个 q 头共享
HEAD_DIM      128                  单头宽度，config 写死，不是 HIDDEN / N_HEADS
INTERMEDIATE  3072                 MLP 中间层宽度
VOCAB         151936               词表大小
RMS_EPS       1e-06                RMSNorm 的 eps
ROPE_THETA    1000000              RoPE 频率基数
SCALING       0.08838834764831845  attention 缩放系数 = HEAD_DIM ** -0.5


### 0.3 证据链：本实验涉及的源码位置

下面的行号从实际加载的模块动态读取，不是手抄的。后续每个模块解剖时会再次引用。

In [6]:
# region 【工具】source_location：把对象定位到源码文件与行号，证据链各处引用它
def source_location(obj) -> str:
    """返回对象在源码中的文件名与起止行号。装饰器包装过的对象会回退到 __wrapped__。"""
    target = inspect.unwrap(obj)
    try:
        lines, start = inspect.getsourcelines(target)
    except (OSError, TypeError) as exc:
        return f'<无法定位: {exc}>'
    return f'{Path(inspect.getfile(target)).name}:{start}-{start + len(lines) - 1}'


EVIDENCE = [
    ('RMSNorm',          Q3.Qwen3RMSNorm),
    ('  └ forward',      Q3.Qwen3RMSNorm.forward),
    ('MLP',              Q3.Qwen3MLP),
    ('  └ forward',      Q3.Qwen3MLP.forward),
    ('RotaryEmbedding',  Q3.Qwen3RotaryEmbedding),
    ('  └ forward',      Q3.Qwen3RotaryEmbedding.forward),
    ('rotate_half',      Q3.rotate_half),
    ('apply_rotary_pos_emb', Q3.apply_rotary_pos_emb),
    ('repeat_kv',        Q3.repeat_kv),
    ('eager_attention_forward', Q3.eager_attention_forward),
    ('Attention',        Q3.Qwen3Attention),
    ('  └ forward',      Q3.Qwen3Attention.forward),
    ('DecoderLayer',     Q3.Qwen3DecoderLayer),
    ('  └ forward',      Q3.Qwen3DecoderLayer.forward),
    ('Qwen3Model',       Q3.Qwen3Model),
    ('  └ forward',      Q3.Qwen3Model.forward),
    ('Qwen3ForCausalLM', Q3.Qwen3ForCausalLM),
    ('  └ forward',      Q3.Qwen3ForCausalLM.forward),
]

for name, obj in EVIDENCE:
    print(f'{name:24s} {source_location(obj)}')
# endregion


RMSNorm                  modeling_qwen3.py:49-67
  └ forward              modeling_qwen3.py:59-64
MLP                      modeling_qwen3.py:70-83
  └ forward              modeling_qwen3.py:81-83
RotaryEmbedding          modeling_qwen3.py:86-137
  └ forward              modeling_qwen3.py:124-137
rotate_half              modeling_qwen3.py:140-144
apply_rotary_pos_emb     modeling_qwen3.py:147-170
repeat_kv                modeling_qwen3.py:173-182
eager_attention_forward  modeling_qwen3.py:185-207
Attention                modeling_qwen3.py:210-280
  └ forward              modeling_qwen3.py:241-280
DecoderLayer             modeling_qwen3.py:283-323
  └ forward              modeling_qwen3.py:294-323
Qwen3Model               modeling_qwen3.py:345-427
  └ forward              modeling_qwen3.py:364-427
Qwen3ForCausalLM         modeling_qwen3.py:430-507
  └ forward              modeling_qwen3.py:446-507


## 1. 入口：tokenizer 只负责把文字变成 input_ids

```text
自然语言
   ↓
tokenizer          ← 本实验不解剖它的内部
   ↓
input_ids          ← 解剖从这里开始
   ↓
Qwen3 前向传播
```

选这句话作为实验输入，是因为它在上下文里给出了 `X 是 Y 的首都` 的模式，
模型只要沿模式补全就应该预测出 `首都`。序列短（9 个 token），
后面 9×9 的 attention 矩阵可以整个打印出来看。

In [7]:
PROMPT = '北京是中国的首都，巴黎是法国的'

encoded = tokenizer(PROMPT, return_tensors='pt')
input_ids = encoded.input_ids.to(DEVICE)
BATCH, SEQ = input_ids.shape

print(f'原文: {PROMPT}')
print(f'input_ids shape: {tuple(input_ids.shape)}   (B={BATCH}, S={SEQ})')
print(f'input_ids: {input_ids[0].tolist()}')
print()
print(f'{"pos":>3s}  {"id":>7s}  {"token":<12s}  decode')
for position, token_id in enumerate(input_ids[0].tolist()):
    token = tokenizer.convert_ids_to_tokens([token_id])[0]
    print(f'{position:>3d}  {token_id:>7d}  {token:<12s}  {tokenizer.decode([token_id])!r}')

原文: 北京是中国的首都，巴黎是法国的
input_ids shape: (1, 9)   (B=1, S=9)
input_ids: [68990, 105196, 9370, 106114, 3837, 106004, 20412, 104328, 9370]

pos       id  token         decode
  0    68990  åĮĹäº¬        '北京'
  1   105196  æĺ¯ä¸ŃåĽ½     '是中国'
  2     9370  çļĦ           '的'
  3   106114  é¦ĸéĥ½        '首都'
  4     3837  ï¼Į           '，'
  5   106004  å·´é»İ        '巴黎'
  6    20412  æĺ¯           '是'
  7   104328  æ³ķåĽ½        '法国'
  8     9370  çļĦ           '的'


上面 `token` 列出现的 `åĮĹäº¬` 这类乱码是 byte-level BPE 的字节表示形式，`decode` 列才是可读文本。
这属于 tokenizer 内部机制，本实验不展开。我们只需要 `input_ids` 这 9 个整数。

# 第一遍：观察真实模型

这一遍完全不写自己的计算。目标是运行官方 forward，并把每个关键节点的中间状态**原样记录下来**，
作为第二遍复现时的比对基准。

记录手段有两类：

1. **官方接口**：`output_hidden_states=True` 给出每层输出，`output_attentions=True` 给出 attention 权重；
2. **forward hooks**：官方接口不暴露的更细的节点（q_proj 输出、q_norm 输出、MLP 中间态、causal mask、
   RoPE 的 cos/sin 等），用 hook 抓。

## 2. 观察工具：只需要记住两个动作

第一遍要抓的中间状态有几百个。为了不让工具本身变成学习负担，整套观察只暴露四样东西：

| 想做的事 | 敲什么 |
|---|---|
| 跑一次官方 forward 并记录全部内部状态 | `qwen = observe(model, input_ids)` |
| 取某一层的某个中间状态 | `qwen.L[0].q_proj` （点出来，能 Tab 补全） |
| 看看它长什么样 | `look(它)` |
| 检查自己算的对不对 | `check(我的, 官方的)` |

**忘了有哪些字段，就敲对象名回车** —— `qwen`、`qwen.L[0]`、`W`、`W.L[0]` 都会打印自己的清单。
这是整套工具唯一需要记住的习惯。

工具还替你兜住了几处只有踩过才知道的细节：hook 参数、每层输出该从哪里取、
attention 节点返回两元组而兄弟节点返回裸张量。其中两条后面会实测：
`output_hidden_states` 的下标语义（§9.1）、为什么判据必须是尺度相对误差（§9.2）。

标着 `# region 【工具】` 的几格是这些工具的实现，可以折起来不看。
要逐行读的是 §5 起的 `my_*` 系列和 §9 的 `MyQwen3*` 系列。


In [8]:
# region 【工具】LAYER_FIELDS + LayerView —— qwen.L[i] 的字段表。敲 qwen.L[0] 就能看到它
# 列表顺序 = 真实计算顺序。
LAYER_FIELDS = [
    ('inp',          '这一层的输入（未归一化，残差记住的就是它）'),
    ('norm1',        'input_layernorm，pre-norm'),
    ('q_proj',       f'{HIDDEN} → {N_HEADS * HEAD_DIM}，{N_HEADS} 头 × {HEAD_DIM}'),
    ('k_proj',       f'{HIDDEN} → {N_KV_HEADS * HEAD_DIM}，{N_KV_HEADS} 头 × {HEAD_DIM}'),
    ('v_proj',       f'{HIDDEN} → {N_KV_HEADS * HEAD_DIM}，{N_KV_HEADS} 头 × {HEAD_DIM}'),
    ('q_norm',       f'在 head_dim={HEAD_DIM} 上归一化'),
    ('k_norm',       '同上。V 没有 norm'),
    ('attn_weights', 'softmax 之后的注意力权重'),
    ('o_proj',       f'{N_HEADS * HEAD_DIM} → {HIDDEN}'),
    ('attn_out',     'Attention 的最终输出（= o_proj 的输出）'),
    ('after_attn',   '第一次残差之后 = inp + attn_out'),
    ('norm2',        'post_attention_layernorm'),
    ('gate',         f'{HIDDEN} → {INTERMEDIATE}'),
    ('up',           f'{HIDDEN} → {INTERMEDIATE}'),
    ('down',         f'{INTERMEDIATE} → {HIDDEN}'),
    ('mlp_out',      'MLP 的最终输出（= down 的输出）'),
    ('out',          '这一层的输出，也是下一层的 inp'),
]


class LayerView:
    """第 index 层的全部中间状态。属性名见 LAYER_FIELDS。"""

    def __init__(self, index):
        self.index = index

    def __repr__(self):
        lines = [f'Layer {self.index} 的中间状态（按计算顺序，可直接 Tab 补全）', '']
        for field, why in LAYER_FIELDS:
            tensor = getattr(self, field, None)
            shape = str(tuple(tensor.shape)) if torch.is_tensor(tensor) else '(未记录)'
            lines.append(f'  .{field:<13s} {shape:<20s} {why}')
        lines.append('')
        lines.append(f'  用法： look(qwen.L[{self.index}].q_proj)   '
                     f'check(我的结果, qwen.L[{self.index}].q_proj)')
        return '\n'.join(lines)
# endregion


In [9]:
# region 【工具】Observed + observe —— 挂 hook、跑一次官方 forward、摘 hook
# 模块路径 → LayerView 字段名。self_attn 和 DecoderLayer 本身单独处理。
_FIELD_OF = {
    'input_layernorm': 'norm1',
    'self_attn.q_proj': 'q_proj',
    'self_attn.k_proj': 'k_proj',
    'self_attn.v_proj': 'v_proj',
    'self_attn.q_norm': 'q_norm',
    'self_attn.k_norm': 'k_norm',
    'self_attn.o_proj': 'o_proj',
    'post_attention_layernorm': 'norm2',
    'mlp.gate_proj': 'gate',
    'mlp.up_proj': 'up',
    'mlp.down_proj': 'down',
    'mlp': 'mlp_out',
}


class Observed:
    """一次官方 forward 的全部内部状态。用 observe(model, input_ids) 得到。"""

    current = None                  # 记住最近一次，look() 解码 token 时要用

    def __init__(self, model, input_ids):
        self.input_ids = input_ids
        self.L = [LayerView(i) for i in range(N_LAYERS)]
        handles = self._mount(model)
        try:
            with torch.no_grad():
                self.official = model(input_ids, output_hidden_states=True,
                                      output_attentions=True, use_cache=False)
        finally:
            for handle in handles:  # 无论成功失败都摘干净，不会污染后续 forward
                handle.remove()
        self.logits = self.official.logits
        # mask 与 cos/sin 由 Qwen3Model 算一次后传给所有 28 层，取第 0 层的即可（§7.2 验证共享）
        self.mask = self.L[0].mask
        self.cos, self.sin = self.L[0].cos, self.L[0].sin
        self.position_ids = self.L[0].position_ids
        self.n_hooks = len(handles)
        Observed.current = self

    def _mount(self, model):
        named = dict(model.named_modules())
        handles = []

        def watch(module, fn):
            # with_kwargs=True 是必需的：attention_mask / position_embeddings 是关键字参数
            handles.append(module.register_forward_hook(
                lambda mod, args, kwargs, output: fn(args, kwargs, output),
                with_kwargs=True))

        def keep(name):
            return lambda args, kwargs, output: setattr(self, name, output)

        watch(model.model.embed_tokens, keep('embed'))
        watch(model.model.norm, keep('final_norm'))

        for index in range(N_LAYERS):
            prefix = f'model.layers.{index}'
            watch(named[prefix], self._layer_hook(index))
            watch(named[f'{prefix}.self_attn'], self._attn_hook(index))
            for path, field in _FIELD_OF.items():
                watch(named[f'{prefix}.{path}'], self._leaf_hook(index, field))
        return handles

    def _leaf_hook(self, index, field):
        return lambda args, kwargs, output: setattr(self.L[index], field, output)

    def _attn_hook(self, index):
        def fn(args, kwargs, output):
            # Qwen3Attention.forward 返回 (attn_output, attn_weights)，在这里就拆开。
            # 对外只有两个普通张量，不需要记得"哪个节点要加 [0]"。
            self.L[index].attn_out, self.L[index].attn_weights = output
        return fn

    def _layer_hook(self, index):
        def fn(args, kwargs, output):
            view = self.L[index]
            view.inp = args[0] if args else kwargs['hidden_states']
            view.out = output                          # DecoderLayer.forward 返回裸张量
            view.after_attn = view.inp + view.attn_out  # 第一次残差，不是任何模块的输出
            view.mask = kwargs['attention_mask']
            view.position_ids = kwargs['position_ids']
            view.cos, view.sin = kwargs['position_embeddings']
        return fn

    def __repr__(self):
        last = N_LAYERS - 1
        rows = [('input_ids', self.input_ids, '输入 token id'),
                ('embed', self.embed, 'embed_tokens 查表，进第 0 层之前'),
                (f'L[0] … L[{last}]', self.L[0].out, f'{N_LAYERS} 层 Decoder Layer，每层保形'),
                ('final_norm', self.final_norm, 'model.norm，最后一次 RMSNorm'),
                ('logits', self.logits, f'lm_head 投到 {VOCAB} 词表')]
        shapes = {tuple(view.out.shape) for view in self.L}
        lines = [f'observe() 已记录 {N_LAYERS} 层内部状态（{self.n_hooks} 个 hook，已全部摘除）', '']
        for name, tensor, why in rows:
            lines.append(f'  {name:<14s} {str(tuple(tensor.shape)):<18s} {why}')
        lines += ['',
                  f'  {N_LAYERS} 层输出形状只有 {len(shapes)} 种：{sorted(shapes)}  → 每层都是保形函数',
                  '',
                  '  看某一层     qwen.L[0]                 （直接敲，会打印字段清单）',
                  '  看某个张量   look(qwen.L[0].q_proj)',
                  '  跨层共享件   qwen.cos  qwen.sin  qwen.mask  qwen.position_ids',
                  '  比对结果     check(我算的, qwen.L[0].q_proj)',
                  '  这一层的权重 W.L[0]                    （§3.1 的权重目录）']
        return '\n'.join(lines)


def observe(model, input_ids):
    """挂 hook → 跑一次官方 forward → 摘 hook，把全部内部状态装进返回值。"""
    return Observed(model, input_ids)
# endregion


In [10]:
qwen = observe(model, input_ids)
qwen

observe() 已记录 28 层内部状态（394 个 hook，已全部摘除）

  input_ids      (1, 9)             输入 token id
  embed          (1, 9, 1024)       embed_tokens 查表，进第 0 层之前
  L[0] … L[27]   (1, 9, 1024)       28 层 Decoder Layer，每层保形
  final_norm     (1, 9, 1024)       model.norm，最后一次 RMSNorm
  logits         (1, 9, 151936)     lm_head 投到 151936 词表

  28 层输出形状只有 1 种：[(1, 9, 1024)]  → 每层都是保形函数

  看某一层     qwen.L[0]                 （直接敲，会打印字段清单）
  看某个张量   look(qwen.L[0].q_proj)
  跨层共享件   qwen.cos  qwen.sin  qwen.mask  qwen.position_ids
  比对结果     check(我算的, qwen.L[0].q_proj)
  这一层的权重 W.L[0]                    （§3.1 的权重目录）

注意 28 层的输出形状**完全一样**，都是 `[1, 9, 1024]`。这是残差结构的直接后果：
每个 Layer 都是一个保形函数，不管内部把维度升到 2048 还是 3072，出口必须回到 1024。


In [11]:
# 忘了某一层有哪些字段时，敲这一行就够了 —— §2 那张表的可执行版本
qwen.L[0]

Layer 0 的中间状态（按计算顺序，可直接 Tab 补全）

  .inp           (1, 9, 1024)         这一层的输入（未归一化，残差记住的就是它）
  .norm1         (1, 9, 1024)         input_layernorm，pre-norm
  .q_proj        (1, 9, 2048)         1024 → 2048，16 头 × 128
  .k_proj        (1, 9, 1024)         1024 → 1024，8 头 × 128
  .v_proj        (1, 9, 1024)         1024 → 1024，8 头 × 128
  .q_norm        (1, 9, 16, 128)      在 head_dim=128 上归一化
  .k_norm        (1, 9, 8, 128)       同上。V 没有 norm
  .attn_weights  (1, 16, 9, 9)        softmax 之后的注意力权重
  .o_proj        (1, 9, 1024)         2048 → 1024
  .attn_out      (1, 9, 1024)         Attention 的最终输出（= o_proj 的输出）
  .after_attn    (1, 9, 1024)         第一次残差之后 = inp + attn_out
  .norm2         (1, 9, 1024)         post_attention_layernorm
  .gate          (1, 9, 3072)         1024 → 3072
  .up            (1, 9, 3072)         1024 → 3072
  .down          (1, 9, 1024)         3072 → 1024
  .mlp_out       (1, 9, 1024)         MLP 的最终输出（= down 的输出）
  .out           (1, 9, 1024)         这一层的输出，

## 3. `look` 和 `check`

**`look(任何东西)`** 只有一个必填参数，自己判断该怎么显示 —— 下面喂三种不同的东西给它看。
完整张量始终保存着，只是不全打印出来；想换 head 或位置再看时才补关键字参数（`look(w, head=5)`）。

**`check(我的, 官方的)`** 判据固定，没有公差可调。每次调用都记进全局清单，
最后 `summary()` 一次性汇总，"哪个节点先出错"由清单顺序直接给出。

另外 `note(名字, 张量)` 登记没有官方对照物的形状节点（`view` / `transpose` / `repeat_kv` 这类），
`summary()` 会连它一起打印。


In [12]:
# region 【工具】note 与 look —— 登记形状节点，以及按类型自动选显示方式
SHAPE_LOG = []      # (名字, 输出形状, 说明)，note() 与 check() 都会往里登记


def note(name, tensor, why=''):
    """登记一个没有官方对照物的中间形状节点（view / transpose / repeat_kv 这类）。"""
    shape = tuple(tensor.shape) if torch.is_tensor(tensor) else tuple(tensor)
    SHAPE_LOG.append((name, shape, why))
    return tensor


def _pad(text, width, align='>'):
    """按终端显示宽度对齐。CJK 字符占 2 列，直接用 len() 会把表格排歪。"""
    shown = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    blanks = ' ' * max(width - shown, 0)
    return blanks + text if align == '>' else text + blanks


def _table(header, rows, indent='  '):
    """按内容自适应列宽拼一张表。三个 repr 共用，省得每处手数列宽。"""
    cols = list(zip(*([header] + rows))) if rows else [(h,) for h in header]
    widths = [max(sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in str(cell))
                  for cell in col) + 2 for col in cols]

    def line(cells):
        parts = [_pad(str(cell), width, '<') for cell, width in zip(cells, widths)]
        return indent + ''.join(parts).rstrip()

    # 分隔线只跟到倒数第二列：最后一列长短不齐，全跟上会拖出一条过长的横线。
    rule = sum(widths[:-1]) + min(widths[-1], 24) - 2
    out = [line(header), indent + '─' * rule]
    out += [line(row) for row in rows]
    return out


def _tokens():
    """当前输入的逐 token 文本，供 look 画注意力网格时当行列标签。"""
    source = Observed.current
    if source is None:
        return None
    return [tokenizer.decode([i]) for i in source.input_ids[0].tolist()]


def _show_grid(weights, head, title):
    """把 [B, heads, S, S] 的注意力权重画成 token × token 网格（值 ×100）。"""
    labels = _tokens() or [str(i) for i in range(weights.shape[-1])]
    seq_len = weights.shape[-1]
    print(f'{title}  (行=query 位置，列=key 位置，值 ×100)')
    print(f'{"":>12s}' + ''.join(_pad(t, 7) for t in labels))
    for i in range(seq_len):
        row = ''.join(f'{weights[0, head, i, j].item() * 100:7.1f}' if j <= i else '      ·'
                      for j in range(seq_len))
        print(f'{_pad(labels[i], 10)} │{row}')


def _stats(tensor):
    """权重的分布摘要。三个数一起看，才发现得了 q_norm / final_norm 的量级异常。"""
    values = tensor.float()
    return (f'mean={values.mean().item():+.4f}  std={values.std().item():.4f}  '
            f'absmax={values.abs().max().item():.4f}')


def _show_matrix(tensor, name, rows, width):
    """打印二维张量的形状与左上角一小块。权重矩阵是 [out_features, in_features]。"""
    out_features, in_features = tensor.shape
    # 用 _pad 而不是 f'{name:<28s}'：名字里有中文时 len() 会算少，表格会排歪。
    print(_pad(name, 28, '<') + _pad(str(tuple(tensor.shape)), 16, '<')
          + f'矩阵 [out={out_features}, in={in_features}] '
            f'→ my_linear(x, w) 内部做 x @ w.T，不必手动转置')
    for row in range(min(rows, out_features)):
        values = ', '.join(f'{v:+.4f}' for v in tensor[row, :width].tolist())
        tail = f'   ← 第 {row} 个输出通道，对应前 {width} 个输入维' if row == 0 else ''
        print(_pad('', 28, '<') + f'w[{row}, :{width}] = [{values}]' + tail)
    print(_pad('', 28, '<') + _stats(tensor))


def _show_vector(tensor, name, width):
    """打印一维张量的形状与开头几个值。RMSNorm 的缩放向量走这里。"""
    print(_pad(name, 28, '<') + _pad(str(tuple(tensor.shape)), 16, '<')
          + '向量 → my_rmsnorm(x, w)，逐元素相乘')
    values = ', '.join(f'{v:+.4f}' for v in tensor[:width].tolist())
    print(_pad('', 28, '<') + f'w[:{width}] = [{values}]')
    print(_pad('', 28, '<') + _stats(tensor))


def _show_window(tensor, name, position, width):
    """打印形状 + 一个小窗口的实际数值。窗口位置随维度自动选。"""
    if tensor.dim() == 3:                                   # [B, S, H]
        window, where = tensor[0, position, :width], f'[0, {position}, :{width}]'
    elif tensor.dim() == 4:                                 # [B, heads, S, D]
        window, where = tensor[0, 0, position, :width], f'[0, 0, {position}, :{width}]'
    else:
        window, where = tensor.flatten()[:width], f'flat[:{width}]'
    values = ', '.join(f'{v:+.4f}' for v in window.tolist())
    print(_pad(name, 28, '<') + _pad(str(tuple(tensor.shape)), 22, '<'))
    print(_pad('', 29, '<') + f'{where} = [{values}]')


# 自带清单式 repr 的对象，look() 直接打印它们的 repr。§3.1 的权重目录会追加进来。
_LOOK_REPR = [Observed, LayerView]


def look(x, name='', head=0, position=-1, width=6, rows=3):
    """看任何东西。按类型和维度自动选显示方式，正常使用不用传后面几个参数。"""
    if isinstance(x, tuple(_LOOK_REPR)):
        print(repr(x))
        return
    if not torch.is_tensor(x):
        print(f'{name or type(x).__name__}: {x!r}')
        return

    if x.dtype in (torch.int32, torch.int64) and x.dim() <= 2:   # token ids
        ids = x.flatten().tolist()
        print(f'{name or "token ids"}  {tuple(x.shape)}')
        for position_index, token_id in enumerate(ids):
            print(f'  {position_index:>3d}  {token_id:>7d}  {tokenizer.decode([token_id])!r}')
        return

    if x.dim() == 4 and x.shape[-1] == x.shape[-2]:              # 注意力权重
        _show_grid(x, head, name or f'attention weights, head {head}')
        return

    if x.dim() == 2:                                             # 权重矩阵 [out, in]
        _show_matrix(x, name or '2D tensor', rows, width)
        return

    if x.dim() == 1:                                             # RMSNorm 权重等一维张量
        _show_vector(x, name or '1D tensor', width)
        return

    _show_window(x, name or f'{x.dim()}D tensor', position, width)
# endregion


In [13]:
# region 【工具】check / record / trace / summary —— 比对与汇总
REL_THRESHOLD = 1e-5        # 尺度相对误差的判定阈值（§9.2 说明为什么不能用固定公差）
CHECKS = []                 # (名字, 是否通过, max_abs, max_rel)，按调用顺序


def check(mine, ref, name=None, quiet=False):
    """比对两个张量。判据固定为 max|差| / max|ref| < 1e-5，没有公差参数可调。"""
    name = name or f'check #{len(CHECKS) + 1}'
    mine, ref = mine.float(), ref.float()
    assert mine.shape == ref.shape, f'{name}: shape 不一致 {tuple(mine.shape)} vs {tuple(ref.shape)}'
    diff = (mine - ref).abs()
    max_abs, mean_abs = diff.max().item(), diff.mean().item()
    scale = ref.abs().max().item()
    rel = max_abs / scale if scale > 0 else 0.0
    ok = rel < REL_THRESHOLD
    CHECKS.append((name, ok, max_abs, rel))
    if not quiet:
        print(f'{"✓" if ok else "✗"} {name:<34s} max_abs={max_abs:.3e}  '
              f'mean_abs={mean_abs:.3e}  max_rel={rel:.3e}')
    return ok


def record(name, ok, rel=0.0):
    """登记一个不是逐元素比对的结论（例如"28 层全部通过"、"预测一致"）。"""
    CHECKS.append((name, ok, 0.0, rel))
    return ok


def trace():
    """打印 note() 到目前为止登记的全部形状节点。"""
    if not SHAPE_LOG:
        return
    width = max(len(row[0]) for row in SHAPE_LOG)
    print(f'shape 变化记录（{len(SHAPE_LOG)} 个节点）')
    print('─' * (width + 46))
    for name, shape, why in SHAPE_LOG:
        print(f'{name:<{width}s}  {shape}' + (f'   # {why}' if why else ''))


def summary():
    """打印 shape 变化记录 + 全部 check 的汇总。"""
    trace()
    print()

    seen, unique = set(), []
    for row in CHECKS:                      # 同名只保留最后一次，避免重复执行留下重复条目
        if row[0] in seen:
            unique = [r for r in unique if r[0] != row[0]]
        seen.add(row[0])
        unique.append(row)

    print(f'共 {len(unique)} 项验证（判据：尺度相对误差 < {REL_THRESHOLD:.0e}）\n')
    print(f'{"项目":<38s} {"max_abs":>11s} {"max_rel":>11s}  结果')
    print('─' * 70)
    for name, ok, max_abs, rel in unique:
        print(f'{name:<38s} {max_abs:>11.3e} {rel:>11.3e}  {"✓" if ok else "✗ 失败"}')
    print('─' * 70)

    failed = [name for name, ok, _, _ in unique if not ok]
    print(f'通过 {len(unique) - len(failed)} / {len(unique)}')
    if failed:
        print(f'首个失败节点: {failed[0]}')
        print(f'全部失败项: {failed}')
    else:
        print('全部通过。从 input_ids 到 Top-K 预测的整条链路已逐节点对齐。')
    return not failed
# endregion


In [14]:
# look 按类型自动选显示方式，同一个函数喂什么都行。
look(input_ids, 'input_ids')                    # 整型 → 逐 token 解码
print()
look(qwen.L[0].out, 'Layer 0 输出')             # 三维 → 形状 + 一个数值窗口
print()
look(qwen.L[0].attn_weights, 'Layer 0 head 0')  # 方阵注意力 → token × token 网格

input_ids  (1, 9)
    0    68990  '北京'
    1   105196  '是中国'
    2     9370  '的'
    3   106114  '首都'
    4     3837  '，'
    5   106004  '巴黎'
    6    20412  '是'
    7   104328  '法国'
    8     9370  '的'

Layer 0 输出                (1, 9, 1024)          
                             [0, -1, :6] = [-0.9747, -0.4370, +0.0096, -0.6557, +0.1166, +0.1212]

Layer 0 head 0  (行=query 位置，列=key 位置，值 ×100)
               北京 是中国     的   首都     ，   巴黎     是   法国     的
      北京 │  100.0      ·      ·      ·      ·      ·      ·      ·      ·
    是中国 │   44.7   55.3      ·      ·      ·      ·      ·      ·      ·
        的 │    3.8   68.6   27.5      ·      ·      ·      ·      ·      ·
      首都 │   39.0    3.8   13.4   43.8      ·      ·      ·      ·      ·
        ， │    0.7    5.6   13.4    0.2   80.1      ·      ·      ·      ·
      巴黎 │    0.7    2.6   41.6    8.8   37.1    9.2      ·      ·      ·
        是 │    0.4    1.4   35.5    0.2   49.7    1.5   11.3      ·      ·
      法国 │    2.2    

In [15]:
# 两边都是官方张量，答案已知：Layer 0 的输出就是 Layer 1 的输入，中间没有别的运算。
# 拿它先看看 check 通过时长什么样 —— 真正的复现比对从 §5 开始。
check(qwen.L[0].out, qwen.L[1].inp, '官方 L0 输出直连 L1 输入');

✓ 官方 L0 输出直连 L1 输入                   max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


### 3.1 权重目录：`W`

上面三个工具管的是**中间状态**（数据流过模型时的样子）。比对还需要另一半：**权重**（模型自己的参数）。
裸路径 `model.model.layers[0].self_attn.q_proj.weight` 能用，问题是记不住有哪些、也记不住形状。
所以再加一份目录，它不封装计算，只把 310 个参数整理成可查的清单：

| 想做的事 | 敲什么 |
|---|---|
| 建目录（只调一次） | `W = params(model)` |
| 看全模型有什么 | `W` |
| 看某一层的 11 个权重 | `W.L[0]` |
| 取一个权重 | `W.L[0].q_proj` |
| 忘了字段叫什么 | `W.find('norm')` |
| 看权重的实际数值 | `look(W.L[0].q_proj)` |

**字段名和 `qwen.L[0]` 完全一样。** `qwen.L[0].q_proj` 是那一层 q_proj 的**输出** `(1, 9, 2048)`，
`W.L[0].q_proj` 是它的**权重** `(2048, 1024)`。一套名字，两边都能用：

```python
check(my_linear(h, W.L[0].q_proj), qwen.L[0].q_proj, 'L0 q_proj')
#                ↑ 权重              ↑ 官方输出
```

清单自带形状、裸路径和「该交给哪个 `my_` 函数」，下面几格敲出来看。字段写错时目录会直接说该敲什么。

取到的是**官方权重本身**，不是副本——这正是 `check` 能对得上的前提，反过来也就不能原地改。
`my_rmsnorm` 在入口断言权重是一维的：两个参数传反时形状会广播成功、**不抛异常**，§5.2 有实测。


In [16]:
# region 【工具】PARAM_FIELDS + LayerParams —— W.L[i] 的字段表与访问护栏
# 权重分两类，用法不同。kind 决定清单里「怎么用」那一列。
PARAM_FIELDS = [
    ('norm1',  'input_layernorm.weight',          'vector'),
    ('q_proj', 'self_attn.q_proj.weight',         'matrix'),
    ('k_proj', 'self_attn.k_proj.weight',         'matrix'),
    ('v_proj', 'self_attn.v_proj.weight',         'matrix'),
    ('q_norm', 'self_attn.q_norm.weight',         'vector'),
    ('k_norm', 'self_attn.k_norm.weight',         'vector'),
    ('o_proj', 'self_attn.o_proj.weight',         'matrix'),
    ('norm2',  'post_attention_layernorm.weight', 'vector'),
    ('gate',   'mlp.gate_proj.weight',            'matrix'),
    ('up',     'mlp.up_proj.weight',              'matrix'),
    ('down',   'mlp.down_proj.weight',            'matrix'),
]
_HOWTO = {'matrix': 'my_linear(x, ·)', 'vector': 'my_rmsnorm(x, ·)'}

# qwen.L[i] 有 17 个字段，这里只有 11 个。差集就是「纯运算的产物，没有权重」那几个，
# 算出来而不是手抄，这样两张表将来改了也不会对不上。
_PARAM_NAMES = [field for field, _, _ in PARAM_FIELDS]
_NO_WEIGHT = [field for field, _ in LAYER_FIELDS if field not in _PARAM_NAMES]


class LayerParams:
    """第 index 层的 11 个权重。字段名与 qwen.L[index] 的中间状态一一对应。"""

    def __init__(self, model, index):
        self._index = index
        layer = model.model.layers[index]
        for field, suffix, _ in PARAM_FIELDS:
            obj = layer
            for part in suffix.split('.'):      # 'self_attn.q_proj.weight' 逐段取下去
                obj = getattr(obj, part)
            setattr(self, field, obj)
        missing = [f for f in _PARAM_NAMES if not torch.is_tensor(getattr(self, f, None))]
        assert not missing, f'Layer {index} 少了权重 {missing}，模型结构与本实验的假设不同'

    def __getattr__(self, name):
        # 只有正常属性找不到时才会走到这里。把几种常见手滑变成会自我解释的报错，
        # 而不是一句光秃秃的 AttributeError。
        if name.startswith('_'):
            raise AttributeError(name)
        index = self.__dict__.get('_index', '?')
        if name in ('bias', 'biases'):
            raise AttributeError(f'本模型没有任何 bias（config 里 attention_bias=false），'
                                 f'W.L[{index}] 只有 weight')
        if name == 'weight':
            raise AttributeError(f'W.L[{index}].字段 取到的已经是权重张量本身，不必再 .weight，'
                                 f'直接写 W.L[{index}].q_proj')
        if name in _NO_WEIGHT:
            raise AttributeError(f'{name!r} 是中间状态、不是权重，在 qwen.L[{index}].{name} 里；'
                                 f'W.L[{index}] 只有 {len(_PARAM_NAMES)} 个权重字段')
        raise AttributeError(f'W.L[{index}] 没有字段 {name!r}。'
                             f'敲 W.L[{index}] 看清单，或 W.find({name!r}) 模糊搜')

    def __repr__(self):
        rows = [(f'.{field}', str(tuple(getattr(self, field).shape)),
                 _HOWTO[kind], suffix)
                for field, suffix, kind in PARAM_FIELDS]
        lines = [f'Layer {self._index} 的 {len(PARAM_FIELDS)} 个权重'
                 f'  （裸路径前缀 model.model.layers[{self._index}].）', '']
        lines += _table(('字段', '形状', '怎么用', '裸路径（可直接复制）'), rows)
        i = self._index
        lines += ['',
                  '  矩阵按 [out, in] 转置存放，交给 my_linear 就不用管 .T。',
                  f'  qwen.L[{i}] 里另有 {len(_NO_WEIGHT)} 个字段是纯中间状态、没有权重：'
                  + '  '.join(_NO_WEIGHT),
                  '',
                  f'  用法： check(my_linear(qwen.L[{i}].norm1, W.L[{i}].q_proj), '
                  f'qwen.L[{i}].q_proj)']
        return '\n'.join(lines)
# endregion


In [17]:
# region 【工具】LayerList + Params + params —— 权重目录本体
class LayerList:
    """W.L 的容器。只认 0..N_LAYERS-1，越界和负数层号都给中文提示。"""

    def __init__(self, items):
        self._items = list(items)

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def __getitem__(self, index):
        if not isinstance(index, int):
            raise TypeError(f'层号要写整数，收到 {index!r}。合法范围 0..{len(self) - 1}')
        if index < 0:
            # 负下标在这里没有好处，只会把 off-by-one 悄悄兜住。最后一层请写明层号。
            raise IndexError(f'不接受负数层号 {index}。最后一层是 W.L[{len(self) - 1}]')
        if index >= len(self):
            raise IndexError(f'层号 {index} 越界。本模型 num_hidden_layers={len(self)}，'
                             f'合法范围 0..{len(self) - 1}')
        return self._items[index]

    def __repr__(self):
        return f'W.L：{len(self)} 层，每层 {len(PARAM_FIELDS)} 个权重。敲 W.L[0] 看清单'


class Params:
    """全模型权重目录。用 params(model) 得到，敲 W 或 W.L[0] 就是清单。"""

    def __init__(self, model):
        self.model = model
        self.L = LayerList(LayerParams(model, i) for i in range(N_LAYERS))
        self.embed = model.model.embed_tokens.weight
        self.final_norm = model.model.norm.weight
        self.lm_head = model.lm_head.weight              # 与 embed 共享同一块内存
        self.inv_freq = model.model.rotary_emb.inv_freq  # buffer，不在 named_parameters() 里

    def _top(self):
        return [('W.embed', self.embed, 'model.model.embed_tokens.weight',
                 '查表取行 → my_embedding'),
                ('W.final_norm', self.final_norm, 'model.model.norm.weight',
                 'my_rmsnorm(x, ·)'),
                ('W.lm_head', self.lm_head, 'model.lm_head.weight',
                 'my_linear(x, ·)  ← 与 embed 同一块内存'),
                ('W.inv_freq', self.inv_freq, 'model.model.rotary_emb.inv_freq',
                 'buffer，不是 parameter')]

    def __repr__(self):
        # 三个数都是现算的,不是写死的:换个模型这张表会跟着变,不会撒谎。
        names = [name for name, _ in self.model.named_parameters()]
        n_bias = sum(1 for name in names if name.endswith('.bias'))
        sample = self.L[0].q_proj
        rows = [(name, str(tuple(tensor.shape)), path, how)
                for name, tensor, path, how in self._top()]
        lines = [f'模型权重目录（named_parameters() 共 {len(names)} 项 '
                 f'= {N_LAYERS} 层 × {len(PARAM_FIELDS)} + 2）',
                 f'dtype={sample.dtype}  device={sample.device}', '']
        lines += _table(('字段', '形状', '裸路径', '怎么用'), rows)
        lines += ['',
                  f'  W.L[0] … W.L[{N_LAYERS - 1}]  每层 {len(PARAM_FIELDS)} 个权重，'
                  f'敲 W.L[0] 看清单；忘了字段名用 W.find("norm")',
                  '',
                  f'  全模型 bias 数量 = {n_bias}（attention_bias=false），找 .bias 会报错。',
                  '  取到的是官方权重本身，不是副本 —— 别原地改。']
        return '\n'.join(lines)

    def find(self, pattern):
        """按名字模糊搜。忘了字段叫什么就 W.find('norm')。"""
        pattern = pattern.lower()
        hits = [(name, tuple(t.shape), path) for name, t, path, _ in self._top()
                if pattern in name.lower() or pattern in path.lower()]
        hits += [(f'W.L[i].{field}', tuple(getattr(self.L[0], field).shape),
                  f'…layers[i].{suffix}')
                 for field, suffix, _ in PARAM_FIELDS
                 if pattern in field.lower() or pattern in suffix.lower()]
        if hits:
            print(f'匹配 {pattern!r} 的 {len(hits)} 个权重字段：\n')
            print('\n'.join(_table(('字段', '形状', '裸路径'),
                                   [(n, str(s), p) for n, s, p in hits])))
        # 搜不到权重时，很可能要找的是中间状态。指过去，而不是只说「没有」。
        states = [field for field, _ in LAYER_FIELDS if pattern in field.lower()]
        if states:
            print(f'\n{pattern!r} 还匹配 {len(states)} 个中间状态（在 qwen.L[i] 里，不是权重）：')
            print('  ' + '  '.join(f'qwen.L[i].{s}' for s in states))
        if not hits and not states:
            print(f'没有匹配 {pattern!r} 的字段。敲 W 看权重目录，敲 qwen.L[0] 看中间状态。')


_LOOK_REPR += [Params, LayerParams, LayerList]   # 这样 look(W) / look(W.L[0]) 也能用


def params(model):
    """把全模型权重整理成一份可查的目录。"""
    return Params(model)
# endregion


In [18]:
W = params(model)
W

模型权重目录（named_parameters() 共 310 项 = 28 层 × 11 + 2）
dtype=torch.float32  device=cpu

  字段          形状            裸路径                           怎么用
  ─────────────────────────────────────────────────────────────────────────────────────
  W.embed       (151936, 1024)  model.model.embed_tokens.weight  查表取行 → my_embedding
  W.final_norm  (1024,)         model.model.norm.weight          my_rmsnorm(x, ·)
  W.lm_head     (151936, 1024)  model.lm_head.weight             my_linear(x, ·)  ← 与 embed 同一块内存
  W.inv_freq    (64,)           model.model.rotary_emb.inv_freq  buffer，不是 parameter

  W.L[0] … W.L[27]  每层 11 个权重，敲 W.L[0] 看清单；忘了字段名用 W.find("norm")

  全模型 bias 数量 = 0（attention_bias=false），找 .bias 会报错。
  取到的是官方权重本身，不是副本 —— 别原地改。

In [19]:
# 忘了某一层有哪些权重、形状是多少，敲这一行
W.L[0]

Layer 0 的 11 个权重  （裸路径前缀 model.model.layers[0].）

  字段     形状          怎么用            裸路径（可直接复制）
  ───────────────────────────────────────────────────────────────
  .norm1   (1024,)       my_rmsnorm(x, ·)  input_layernorm.weight
  .q_proj  (2048, 1024)  my_linear(x, ·)   self_attn.q_proj.weight
  .k_proj  (1024, 1024)  my_linear(x, ·)   self_attn.k_proj.weight
  .v_proj  (1024, 1024)  my_linear(x, ·)   self_attn.v_proj.weight
  .q_norm  (128,)        my_rmsnorm(x, ·)  self_attn.q_norm.weight
  .k_norm  (128,)        my_rmsnorm(x, ·)  self_attn.k_norm.weight
  .o_proj  (1024, 2048)  my_linear(x, ·)   self_attn.o_proj.weight
  .norm2   (1024,)       my_rmsnorm(x, ·)  post_attention_layernorm.weight
  .gate    (3072, 1024)  my_linear(x, ·)   mlp.gate_proj.weight
  .up      (3072, 1024)  my_linear(x, ·)   mlp.up_proj.weight
  .down    (1024, 3072)  my_linear(x, ·)   mlp.down_proj.weight

  矩阵按 [out, in] 转置存放，交给 my_linear 就不用管 .T。
  qwen.L[0] 里另有 6 个字段是纯中间状态、没有权重：inp  attn_weights  attn_out

In [20]:
# 只记得名字里带 norm，不确定完整字段名
W.find('norm')

匹配 'norm' 的 5 个权重字段：

  字段           形状     裸路径
  ──────────────────────────────────────────────
  W.final_norm   (1024,)  model.model.norm.weight
  W.L[i].norm1   (1024,)  …layers[i].input_layernorm.weight
  W.L[i].q_norm  (128,)   …layers[i].self_attn.q_norm.weight
  W.L[i].k_norm  (128,)   …layers[i].self_attn.k_norm.weight
  W.L[i].norm2   (1024,)  …layers[i].post_attention_layernorm.weight

'norm' 还匹配 4 个中间状态（在 qwen.L[i] 里，不是权重）：
  qwen.L[i].norm1  qwen.L[i].q_norm  qwen.L[i].k_norm  qwen.L[i].norm2


In [21]:
# 清单只给形状，想看权重的实际数值就交给 look：
# 矩阵显示左上角一小块并注明 [out, in]，向量显示开头几个值，末行都带分布摘要。
look(W.L[0].q_proj, 'q_proj 权重')
print()
look(W.L[0].norm1, 'input_layernorm 权重')

q_proj 权重                 (2048, 1024)    矩阵 [out=2048, in=1024] → my_linear(x, w) 内部做 x @ w.T，不必手动转置
                            w[0, :6] = [+0.0059, -0.0042, -0.0137, +0.0197, +0.0142, -0.0013]   ← 第 0 个输出通道，对应前 6 个输入维
                            w[1, :6] = [-0.0264, +0.0089, -0.0012, +0.0220, +0.0011, +0.0027]
                            w[2, :6] = [+0.0220, +0.0032, -0.0033, +0.0091, +0.0028, -0.0067]
                            mean=-0.0000  std=0.0317  absmax=0.6602

input_layernorm 权重        (1024,)         向量 → my_rmsnorm(x, w)，逐元素相乘
                            w[:6] = [+0.1377, +0.7109, +0.5977, +0.7227, +0.2070, +0.6367]
                            mean=+0.1773  std=0.0753  absmax=1.0469


## 4. 顶层 Shape Trace：一次 forward 的骨架

先不打开任何模块，只看数据在五个顶层阶段之间的形状变化。这是实验 1 那张图的数值版本。
`note()` 把节点登记进 shape 记录，`trace()` 随时把已登记的内容打印出来——
§8 解剖完 Layer 0 之后再调一次，就能看到骨架和细节接在同一条记录上。

In [22]:
note('input_ids', input_ids, 'B=1, S=9 的整数张量')
note('embed_tokens', qwen.embed, '查表：行号 → 1024 维向量')
note('layers.0', qwen.L[0].out, '保形')
note('layers.1', qwen.L[1].out, '保形  ⋯ 中间 26 层省略')
note(f'layers.{N_LAYERS - 1}', qwen.L[N_LAYERS - 1].out, '保形')
note('final_norm', qwen.final_norm, 'RMSNorm，不改形状')
note('lm_head', qwen.logits, f'1024 → {VOCAB} 词表打分')

trace()

shape 变化记录（7 个节点）
──────────────────────────────────────────────────────────
input_ids     (1, 9)   # B=1, S=9 的整数张量
embed_tokens  (1, 9, 1024)   # 查表：行号 → 1024 维向量
layers.0      (1, 9, 1024)   # 保形
layers.1      (1, 9, 1024)   # 保形  ⋯ 中间 26 层省略
layers.27     (1, 9, 1024)   # 保形
final_norm    (1, 9, 1024)   # RMSNorm，不改形状
lm_head       (1, 9, 151936)   # 1024 → 151936 词表打分


# 第二遍：自己复现

原则是**能自己实现就自己实现**，但不重复制造 PyTorch。

**自己实现**：Embedding、Linear、RMSNorm、RoPE、GQA、Attention、causal mask、MLP、Residual、Decoder Layer。

**直接用 PyTorch 基础算子**：`matmul`、`reshape`、`view`、`transpose`、`softmax`、`exp`、`rsqrt`、`sigmoid` 等。

Linear 和 RMSNorm 是第一次完整实现，之后所有地方复用自己的这一份实现。

## 5. 最基础的两块积木

### 5.1 Linear（无 bias）

`config.attention_bias = False`，且 MLP 的三个投影在源码里都是 `bias=False`，
所以本模型**所有** Linear 都没有 bias。计算就是一次矩阵乘：

$$\mathrm{Linear}(x) = x W^\top$$

`nn.Linear` 的权重形状是 `[out_features, in_features]`，所以要转置后右乘。

In [23]:
def my_linear(x, weight):
    """无 bias 的线性层。weight: [out_features, in_features]"""
    assert weight.dim() == 2, (
        f'my_linear 的第二个参数要是二维权重矩阵，收到 {weight.dim()} 维 '
        f'{tuple(weight.shape)}。一维的是 RMSNorm 系数，该用 my_rmsnorm')
    assert x.shape[-1] == weight.shape[1], (
        f'形状对不上：x 最后一维 {x.shape[-1]}，而 weight 的 in_features 是 '
        f'{weight.shape[1]}（weight 形状 {tuple(weight.shape)} 是 [out, in]）')
    return x @ weight.T


# 权重从目录取。W.L[0].q_proj 就是 model.model.layers[0].self_attn.q_proj.weight，
# 同一个张量，只是不必手敲那一长串（敲 W.L[0] 可以看到裸路径）。
_probe_w = W.L[0].q_proj
print(f'q_proj 权重形状: {tuple(_probe_w.shape)}  (out=2048, in=1024)')
print(f'与裸路径是同一对象: '
      f'{_probe_w is model.model.layers[0].self_attn.q_proj.weight}')
check(my_linear(qwen.L[0].norm1, _probe_w), qwen.L[0].q_proj, 'my_linear vs q_proj')

q_proj 权重形状: (2048, 1024)  (out=2048, in=1024)
与裸路径是同一对象: True
✓ my_linear vs q_proj                max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 5.2 RMSNorm

源码 `Qwen3RMSNorm.forward`（见 §0.3 证据链）的计算顺序是：

$$\bar{x} = x \cdot \frac{1}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}, \qquad
\mathrm{RMSNorm}(x) = w \odot \bar{x}$$

严格按代码顺序，有三个容易写错的细节：

1. **先转 float32 再算**，最后才转回输入 dtype；
2. `weight` 的乘法发生在**转回 dtype 之后**，不是在 float32 里乘；
3. 用 `rsqrt(variance + eps)`，eps 在**根号内部**，不是外部；
4. 这里的 variance 是**平方的均值**，不减均值（这是 RMSNorm 与 LayerNorm 的区别）。

In [24]:
def my_rmsnorm(x, weight, eps=RMS_EPS):
    """严格按 Qwen3RMSNorm.forward 的顺序实现。归一化沿最后一维进行。"""
    assert weight.dim() == 1, (
        f'my_rmsnorm 的第二个参数要是一维缩放系数，收到 {weight.dim()} 维 '
        f'{tuple(weight.shape)}。参数顺序是 my_rmsnorm(输入, 权重)')
    assert x.shape[-1] == weight.shape[0], (
        f'形状对不上：x 最后一维 {x.shape[-1]}，权重长度 {weight.shape[0]}')
    input_dtype = x.dtype
    x = x.to(torch.float32)                              # 1. 升到 float32
    variance = x.pow(2).mean(-1, keepdim=True)           # 2. 平方的均值，不减均值
    x = x * torch.rsqrt(variance + eps)                  # 3. eps 在根号内
    return weight * x.to(input_dtype)                    # 4. 先转回 dtype，再乘 weight


check(my_rmsnorm(qwen.L[0].inp, W.L[0].norm1),
      qwen.L[0].norm1, 'my_rmsnorm vs input_layernorm')
# post_attention_layernorm 的输入是"第一次残差之后"的中间态。
# 它不是任何模块的直接输出，工具已经替我们拼好了：qwen.L[0].after_attn = inp + attn_out。
check(my_rmsnorm(qwen.L[0].after_attn, W.L[0].norm2),
      qwen.L[0].norm2, 'my_rmsnorm vs post_attn_norm')

print()
print('参数传反 my_rmsnorm(权重, 输入) 会怎样：')
try:
    my_rmsnorm(W.L[0].norm1, qwen.L[0].inp)
    print('  居然没报错 ← 断言失效了')
except AssertionError as e:
    print(f'  断言拦住了：{str(e).splitlines()[0].strip()}')
print('  没有这条断言的话，(1024,) 与 (1, 9, 1024) 会广播成功 —— 形状对、不报错、结果偏 4%。')


✓ my_rmsnorm vs input_layernorm      max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_rmsnorm vs post_attn_norm       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

参数传反 my_rmsnorm(权重, 输入) 会怎样：
  断言拦住了：my_rmsnorm 的第二个参数要是一维缩放系数，收到 3 维 (1, 9, 1024)。参数顺序是 my_rmsnorm(输入, 权重)
  没有这条断言的话，(1024,) 与 (1, 9, 1024) 会广播成功 —— 形状对、不报错、结果偏 4%。


### 5.3 SiLU 激活

`config.hidden_act = 'silu'`，即 $\mathrm{SiLU}(x) = x \cdot \sigma(x) = \dfrac{x}{1 + e^{-x}}$。
我们用 `torch.sigmoid` 这个基础算子拼出来，不调 `nn.functional.silu`。

In [25]:
def my_silu(x):
    return x * torch.sigmoid(x)


_t = torch.randn(4, 8, dtype=DTYPE)
check(my_silu(_t), torch.nn.functional.silu(_t), 'my_silu vs F.silu')

✓ my_silu vs F.silu                  max_abs=5.960e-08  mean_abs=8.149e-09  max_rel=4.212e-08


True

## 6. Embedding：一次按行查表

```text
              input_ids [B, S]              整数，取值范围 [0, 151936)
                   │
                   │  以 token id 作为行号，从权重矩阵取行
                   ▼
   embed_tokens.weight [151936, 1024]
                   │
                   ▼
        inputs_embeds [B, S, 1024]
```

`nn.Embedding` 的 forward 没有矩阵乘、没有激活，就是一次索引。所以"自己实现"它等价于
`weight[input_ids]`。下面顺带验证一件事：**这张表就是最后 lm_head 用的那张表**。

In [26]:
def my_embedding(ids, weight):
    """按行号取行。等价于 nn.Embedding.forward（无 padding_idx 参与时）。"""
    return weight[ids]


EMBED_WEIGHT = W.embed          # = model.model.embed_tokens.weight
my_embeds = my_embedding(input_ids, EMBED_WEIGHT)

check(my_embeds, qwen.embed, 'my_embedding vs embed_tokens')
check(my_embeds, qwen.L[0].inp, 'my_embedding vs Layer 0 输入')

print()
print(f'W.embed    (embed_tokens.weight)  {tuple(EMBED_WEIGHT.shape)}')
print(f'W.lm_head  (lm_head.weight)       {tuple(W.lm_head.shape)}')
# 要证明的是官方模型里 tie_word_embeddings 生效,所以右边一律用官方裸路径,
# 不拿目录自己的两个字段互证(那只能说明 params() 写对了)。
_official_lm_head = model.lm_head.weight
print(f'W.lm_head 取到的就是它:  {W.lm_head is _official_lm_head}')
print(f'是同一个张量对象:       {EMBED_WEIGHT is _official_lm_head}')
print(f'共享同一块内存:         {EMBED_WEIGHT.data_ptr() == _official_lm_head.data_ptr()}')
print(f'_tied_weights_keys:    {type(model)._tied_weights_keys}')
print()
_embed_params = VOCAB * HIDDEN
_total = sum(p.numel() for p in model.parameters())
print(f'这张表的参数量: {_embed_params:,} = 总参数 {_total:,} 的 {_embed_params / _total:.1%}')

✓ my_embedding vs embed_tokens       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_embedding vs Layer 0 输入         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

W.embed    (embed_tokens.weight)  (151936, 1024)
W.lm_head  (lm_head.weight)       (151936, 1024)
W.lm_head 取到的就是它:  True
是同一个张量对象:       True
共享同一块内存:         True
_tied_weights_keys:    {'lm_head.weight': 'model.embed_tokens.weight'}

这张表的参数量: 155,582,464 = 总参数 596,049,920 的 26.1%


`tie_word_embeddings: true` 的含义在这里变得具体：**同一张 `[151936, 1024]` 的矩阵被用了两次**。

- 入口（Embedding）当字典查：行号 → 向量；
- 出口（LM Head）当打分器用：把 hidden state 和 151936 行逐行做内积。

这也是为什么 `model.safetensors` 里存了 310 个张量，却**没有** `lm_head.weight`——它根本不需要存。
这一张表占了模型总参数的四分之一以上。

In [27]:
# 逐个 token 确认"查表"就是字面意义的取行
for position in [0, SEQ - 1]:
    token_id = input_ids[0, position].item()
    row = EMBED_WEIGHT[token_id]
    got = my_embeds[0, position]
    same = torch.equal(row, got)
    text = tokenizer.decode([token_id])
    print(f'pos {position}  id={token_id:<7d} {text!r:<6s} '
          f'embeds[0,{position}] 是否等于 weight[{token_id}]: {same}')
    print(f'         前 6 维 = [{", ".join(f"{v:+.5f}" for v in row[:6].tolist())}]')

pos 0  id=68990   '北京'   embeds[0,0] 是否等于 weight[68990]: True
         前 6 维 = [-0.03198, -0.04858, +0.03345, +0.04492, -0.03296, -0.04077]
pos 8  id=9370    '的'    embeds[0,8] 是否等于 weight[9370]: True
         前 6 维 = [-0.01941, -0.00482, -0.04858, -0.01636, -0.00824, +0.02405]


## 7. 进入 Layer 之前：两样全局预备件

看源码 `Qwen3Model.forward` 会发现，进入 28 层循环之前先算好了两样东西，
然后**原样传给每一层**，28 层共用，不重复计算：

```text
inputs_embeds
     │
     ├──→ position_ids ──→ rotary_emb ──→ (cos, sin)   ─┐
     │                                                  ├─→ 传给全部 28 层
     └──→ create_causal_mask ──→ attention_mask        ─┘
```

这一点很容易被忽略：RoPE 的 cos/sin **不在 Attention 内部计算**，而是模型级别算一次。

In [28]:
# Qwen3Model.forward 里的 position_ids：没传就是 0..S-1
my_position_ids = torch.arange(SEQ, device=DEVICE).unsqueeze(0)
print(f'position_ids {tuple(my_position_ids.shape)} = {my_position_ids[0].tolist()}')

# 官方实际用的 position_ids（工具已从 Decoder Layer 的 kwargs 抓好）
check(my_position_ids.float(), qwen.position_ids.float(), 'my_position_ids')

position_ids (1, 9) = [0, 1, 2, 3, 4, 5, 6, 7, 8]
✓ my_position_ids                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 7.1 RoPE：cos / sin 表

RoPE 的目标是把"位置"编码成一个旋转。频率向量（`inv_freq`）只依赖 head_dim 和 theta：

$$\theta_j = \frac{1}{\text{base}^{\,2j/d}}, \qquad j = 0, 1, \dots, \frac{d}{2}-1$$

其中 $d = 128$（head_dim），base = `rope_theta` = 1000000。注意源码里
`torch.arange(0, dim, 2) / dim` 得到的是 $2j/d$，所以 `inv_freq` 长度是 64。

然后与位置做外积，再把结果**复制一份拼接**成 128 维：

$$\text{freqs}[p, j] = p \cdot \theta_j \quad (\text{形状 } S \times 64), \qquad
\text{emb} = [\text{freqs}, \text{freqs}] \quad (S \times 128)$$

$$\cos = \cos(\text{emb}), \qquad \sin = \sin(\text{emb})$$

拼接这一步是为了配合后面 `rotate_half` 的实现方式。

In [29]:
def my_rope_tables(position_ids, head_dim=HEAD_DIM, base=ROPE_THETA, dtype=DTYPE):
    """复现 Qwen3RotaryEmbedding：返回 (cos, sin)，形状 [B, S, head_dim]。"""
    # inv_freq: [head_dim/2]，全程 float32
    exponent = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
    inv_freq = 1.0 / (base ** exponent)

    # 外积：[B, head_dim/2, 1] @ [B, 1, S] -> [B, head_dim/2, S] -> transpose -> [B, S, head_dim/2]
    inv_freq_expanded = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
    positions_expanded = position_ids[:, None, :].float()
    freqs = (inv_freq_expanded @ positions_expanded).transpose(1, 2)

    emb = torch.cat((freqs, freqs), dim=-1)      # [B, S, head_dim]
    return emb.cos().to(dtype), emb.sin().to(dtype), inv_freq


my_cos, my_sin, my_inv_freq = my_rope_tables(my_position_ids)

print(f'inv_freq {tuple(my_inv_freq.shape)}  前4 = '
      f'[{", ".join(f"{v:.3e}" for v in my_inv_freq[:4].tolist())}]')
print(f'         后4 = [{", ".join(f"{v:.3e}" for v in my_inv_freq[-4:].tolist())}]')
print(f'cos/sin  {tuple(my_cos.shape)}')
print()
check(my_cos, qwen.cos, 'my_rope cos')
check(my_sin, qwen.sin, 'my_rope sin')
check(my_inv_freq, W.inv_freq, 'my_inv_freq')

inv_freq (64,)  前4 = [1.000e+00, 8.058e-01, 6.494e-01, 5.233e-01]
         后4 = [2.371e-06, 1.911e-06, 1.540e-06, 1.241e-06]
cos/sin  (1, 9, 128)

✓ my_rope cos                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_rope sin                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ my_inv_freq                        max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [30]:
# 观察窗口：cos 表的前 4 个位置 × 前 3 个频率通道
print('cos[0, position, channel] 的一角：')
print(f'{"":>8s}' + ''.join(f'ch{c:<10d}' for c in range(3)))
for position in range(4):
    row = ''.join(f'{my_cos[0, position, c].item():+11.6f}' for c in range(3))
    print(f'pos {position:<3d} {row}')
print()
print('ch0 频率最高（相邻位置差异大），高编号通道频率极低（长距离才有区分度）：')
for channel in [0, 32, 63]:
    values = ', '.join(f'{my_cos[0, p, channel].item():+.6f}' for p in range(SEQ))
    print(f'  ch{channel:<3d} 沿位置变化 = [{values}]')

cos[0, position, channel] 的一角：
        ch0         ch1         ch2         
pos 0     +1.000000  +1.000000  +1.000000
pos 1     +0.540302  +0.692504  +0.796458
pos 2     -0.416147  -0.040877  +0.268690
pos 3     -0.989992  -0.749119  -0.368457

ch0 频率最高（相邻位置差异大），高编号通道频率极低（长距离才有区分度）：
  ch0   沿位置变化 = [+1.000000, +0.540302, -0.416147, -0.989992, -0.653644, +0.283662, +0.960170, +0.753902, -0.145500]
  ch32  沿位置变化 = [+1.000000, +1.000000, +0.999998, +0.999996, +0.999992, +0.999987, +0.999982, +0.999976, +0.999968]
  ch63  沿位置变化 = [+1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000, +1.000000]


### 7.2 Causal Mask

因果掩码保证位置 $i$ 只能看到 $j \le i$ 的位置。eager 路径下它是一个**加性** mask：
允许的位置填 0，禁止的位置填一个极小的数（`torch.finfo(float32).min`），
加到 attention 分数上之后，softmax 会把这些位置压到 0。

```text
        j=0   1    2   ...          ← 被看的位置（key）
 i=0  [  0  -inf -inf ...  ]
 i=1  [  0    0  -inf ...  ]        ← 看的位置（query）
 i=2  [  0    0    0  ...  ]
```

In [31]:
def my_causal_mask(seq_len, dtype=DTYPE, device=DEVICE):
    """加性因果掩码，形状 [1, 1, S, S]，可广播到 [B, heads, S, S]。"""
    blocked = torch.finfo(dtype).min
    positions = torch.arange(seq_len, device=device)
    # query 位置 i 只能看 key 位置 j <= i
    allowed = positions[:, None] >= positions[None, :]
    mask = torch.where(allowed, torch.zeros((), dtype=dtype, device=device),
                       torch.full((), blocked, dtype=dtype, device=device))
    return mask[None, None, :, :]


my_mask = my_causal_mask(SEQ)

print(f'官方 mask: {tuple(qwen.mask.shape)}  dtype={qwen.mask.dtype}')
print(f'屏蔽值:    {qwen.mask.min().item():.6e}')
print(f'等于 torch.finfo(float32).min: '
      f'{qwen.mask.min().item() == torch.finfo(torch.float32).min}')
print()
check(my_mask, qwen.mask, 'my_causal_mask')
print()
print('mask[0,0] 的 0/-inf 结构（0 表示可见，· 表示屏蔽）：')
for i in range(SEQ):
    row = ' '.join('0' if qwen.mask[0, 0, i, j] == 0 else '·' for j in range(SEQ))
    print(f'  i={i}  {row}')

官方 mask: (1, 1, 9, 9)  dtype=torch.float32
屏蔽值:    -3.402823e+38
等于 torch.finfo(float32).min: True

✓ my_causal_mask                     max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

mask[0,0] 的 0/-inf 结构（0 表示可见，· 表示屏蔽）：
  i=0  0 · · · · · · · ·
  i=1  0 0 · · · · · · ·
  i=2  0 0 0 · · · · · ·
  i=3  0 0 0 0 · · · · ·
  i=4  0 0 0 0 0 · · · ·
  i=5  0 0 0 0 0 0 · · ·
  i=6  0 0 0 0 0 0 0 · ·
  i=7  0 0 0 0 0 0 0 0 ·
  i=8  0 0 0 0 0 0 0 0 0


所有 28 层共用同一个 mask 对象和同一组 cos/sin，下面确认这一点。

In [32]:
mask_shared = all(qwen.L[i].mask is qwen.mask for i in range(N_LAYERS))
rope_shared = all(qwen.L[i].cos is qwen.cos for i in range(N_LAYERS))
print(f'28 层共用同一个 attention_mask 对象: {mask_shared}')
print(f'28 层共用同一组 (cos, sin) 对象:      {rope_shared}')
record('28 层共用 mask 与 RoPE 表', mask_shared and rope_shared)

28 层共用同一个 attention_mask 对象: True
28 层共用同一组 (cos, sin) 对象:      True


True

## 8. 完整解剖 Layer 0

28 层结构完全相同，所以只对第 0 层做逐步解剖。后续 27 层用同样的逻辑批量计算并全部验证，
但不重复写 27 遍说明。

```text
                    Layer Input  [B, S, 1024]
                          │
             ┌────────────┤ residual
             │            ▼
             │      input_layernorm (RMSNorm)      [B, S, 1024]
             │            ▼
             │      ┌─────────────────────────┐
             │      │      Attention          │
             │      │  q/k/v_proj → q/k_norm  │
             │      │  → RoPE → GQA → QKᵀ     │
             │      │  → mask → softmax → ×V  │
             │      │  → o_proj               │
             │      └─────────────────────────┘    [B, S, 1024]
             │            ▼
             └──────────► ⊕  residual add          [B, S, 1024]
                          │
             ┌────────────┤ residual
             │            ▼
             │      post_attention_layernorm       [B, S, 1024]
             │            ▼
             │      ┌─────────────────────────┐
             │      │          MLP            │
             │      │  gate_proj  up_proj     │    [B, S, 3072]
             │      │  SiLU(gate) * up        │
             │      │  down_proj              │    [B, S, 1024]
             │      └─────────────────────────┘
             │            ▼
             └──────────► ⊕  residual add
                          │
                    Layer Output [B, S, 1024]
```

源码依据：`Qwen3DecoderLayer.forward`（§0.3 证据链给出行号）。
注意两个 RMSNorm 都在**子模块之前**（pre-norm），残差加的是**未归一化**的输入。

In [33]:
LAYER = 0
L0 = qwen.L[LAYER]          # 这一层的官方中间状态，敲 L0 回车看字段清单
P0 = W.L[LAYER]             # 这一层的 11 个权重，敲 P0 回车看字段清单
# 两边字段同名：L0.q_proj 是输出，P0.q_proj 是权重

# ── Layer 输入 ──
x_in = L0.inp
note('layer input', x_in, '= embedding 输出')
residual_1 = x_in                      # 残差记住的是未归一化的输入

# ── 8.1 input_layernorm ──
h = my_rmsnorm(x_in, P0.norm1)
note('input_layernorm', h, 'RMSNorm，形状不变')
check(h, L0.norm1, f'L{LAYER} input_layernorm')

look(x_in, 'layer input')
look(h, 'after input_layernorm')

✓ L0 input_layernorm                 max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
layer input                 (1, 9, 1024)          
                             [0, -1, :6] = [-0.0194, -0.0048, -0.0486, -0.0164, -0.0082, +0.0240]
after input_layernorm       (1, 9, 1024)          
                             [0, -1, :6] = [-0.1020, -0.1308, -1.1080, -0.4511, -0.0651, +0.5843]


### 8.2 Q / K / V 投影

三个投影的输出维度不一样，这是 GQA（Grouped Query Attention）的起点：

| 投影 | weight 形状 | 输出 | 含义 |
|---|---|---|---|
| `q_proj` | `[2048, 1024]` | `[B, S, 2048]` | 16 个 query 头 × 128 |
| `k_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 key 头 × 128 |
| `v_proj` | `[1024, 1024]` | `[B, S, 1024]` | **8** 个 value 头 × 128 |

query 头数是 kv 头数的 2 倍。注意 `q_proj` 把 1024 维**升到了 2048**，
比 hidden_size 还大——这是 Qwen3 的选择，`head_dim=128` 是配置里写死的，不是 `hidden_size / n_heads`。

In [34]:
q_flat = my_linear(h, P0.q_proj)
k_flat = my_linear(h, P0.k_proj)
v_flat = my_linear(h, P0.v_proj)

note('q_proj', q_flat, '16 头 × 128')
note('k_proj', k_flat, '8 头 × 128')
note('v_proj', v_flat, '8 头 × 128')

check(q_flat, L0.q_proj, f'L{LAYER} q_proj')
check(k_flat, L0.k_proj, f'L{LAYER} k_proj')
check(v_flat, L0.v_proj, f'L{LAYER} v_proj')

✓ L0 q_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 k_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 v_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 8.3 reshape 成多头，然后做 q_norm / k_norm

**这一步是 Qwen3 与 Llama 最关键的结构差异。** 源码里这三行把好几个操作压在了一起：

```python
query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
key_states   = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1, 2)
value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
```

拆开看，真实顺序是：

```text
q_proj 输出  [B, S, 2048]
    ↓ view(B, S, -1, 128)
             [B, S, 16, 128]
    ↓ q_norm  ← RMSNorm 作用在最后一维 head_dim=128 上，不是 1024！
             [B, S, 16, 128]
    ↓ transpose(1, 2)
             [B, 16, S, 128]
```

三个要点：

1. norm 在 **reshape 之后**做，所以归一化的单位是**每个头的 128 维向量**，不是整个 2048；
2. `q_norm.weight` 的形状是 `[128]`，**16 个头共享同一组 128 个缩放参数**；
3. **V 没有 norm**，只有 Q 和 K 有。

In [35]:
print(f'q_norm 权重形状: {tuple(P0.q_norm.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_HEADS} 个 q 头共享')
print(f'k_norm 权重形状: {tuple(P0.k_norm.shape)}  '
      f'← 长度 {HEAD_DIM}，被 {N_KV_HEADS} 个 kv 头共享')
print(f'v 有 norm 吗: {hasattr(model.model.layers[LAYER].self_attn, "v_norm")}')
print()

# view: 把最后一维拆成 (头数, head_dim)
q_heads = q_flat.view(BATCH, SEQ, N_HEADS, HEAD_DIM)
k_heads = k_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)
v_heads = v_flat.view(BATCH, SEQ, N_KV_HEADS, HEAD_DIM)
note('q view', q_heads, '2048 拆成 16×128')
note('k view', k_heads, '1024 拆成 8×128')

# q_norm / k_norm 在 head_dim 上做 RMSNorm
q_normed = my_rmsnorm(q_heads, P0.q_norm)
k_normed = my_rmsnorm(k_heads, P0.k_norm)
note('q_norm', q_normed, '在 head_dim=128 上归一化')
note('k_norm', k_normed, 'V 不做 norm')

check(q_normed, L0.q_norm, f'L{LAYER} q_norm')
check(k_normed, L0.k_norm, f'L{LAYER} k_norm')

q_norm 权重形状: (128,)  ← 长度 128，被 16 个 q 头共享
k_norm 权重形状: (128,)  ← 长度 128，被 8 个 kv 头共享
v 有 norm 吗: False

✓ L0 q_norm                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 k_norm                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [36]:
# transpose 把头维提到前面，之后所有 attention 计算都在 [B, heads, S, head_dim] 上做
q = q_normed.transpose(1, 2)
k = k_normed.transpose(1, 2)
v = v_heads.transpose(1, 2)

note('q transpose(1,2)', q, '头维提前')
note('k transpose(1,2)', k)
note('v transpose(1,2)', v)

print('norm 前后对比（head 0，最后一个位置，前 6 维）：')
look(q_heads.transpose(1, 2), 'q 未 norm')
look(q, 'q 已 norm')
print()
print('验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1')
_h0 = q_heads[0, -1, 0].float()
print(f'  norm 前 RMS = {_h0.pow(2).mean().sqrt().item():.4f}')
_h0n = (q_normed[0, -1, 0] / P0.q_norm).float()
print(f'  norm 后（除掉 weight）RMS = {_h0n.pow(2).mean().sqrt().item():.6f}')

norm 前后对比（head 0，最后一个位置，前 6 维）：
q 未 norm                   (1, 16, 9, 128)       
                             [0, 0, -1, :6] = [+0.0292, +0.1119, -0.0286, -0.1662, -0.0711, +0.1085]
q 已 norm                   (1, 16, 9, 128)       
                             [0, 0, -1, :6] = [+0.5642, +0.5918, +0.0894, -1.2054, -0.7853, +0.7144]

验证 q_norm 真的是逐头归一化：head 0 的 128 维向量，其均方根应该接近 1
  norm 前 RMS = 0.2349
  norm 后（除掉 weight）RMS = 0.999991


### 8.4 应用 RoPE

$$q' = q \odot \cos + \mathrm{rotate\_half}(q) \odot \sin$$

其中 `rotate_half` 把 128 维**前后对半切开**再交叉取负：

$$\mathrm{rotate\_half}([x_1, x_2]) = [-x_2, x_1], \qquad x_1 = x[:64],\ x_2 = x[64:]$$

这与"把相邻两维配成一对做二维旋转"的经典写法在数学上等价，但**维度配对方式不同**：
这里配对的是 $(i, i+64)$，不是 $(2i, 2i+1)$。这也解释了 §7.1 为什么要把 freqs 复制拼接成 128 维——
`cos` 的第 $i$ 维和第 $i+64$ 维是同一个角度。

cos/sin 形状是 `[B, S, 128]`，要 `unsqueeze(1)` 变成 `[B, 1, S, 128]` 才能广播到 `[B, heads, S, 128]`。
**Q 和 K 都要转，V 不转。**

In [37]:
def my_rotate_half(x):
    """把最后一维对半切开，交叉取负。"""
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((-x2, x1), dim=-1)


def my_apply_rope(q, k, cos, sin):
    """cos/sin: [B, S, head_dim] -> unsqueeze 到 [B, 1, S, head_dim] 广播。"""
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    q_out = q * cos + my_rotate_half(q) * sin
    k_out = k * cos + my_rotate_half(k) * sin
    return q_out, k_out


check(my_rotate_half(q), Q3.rotate_half(q), 'my_rotate_half')

q_rope, k_rope = my_apply_rope(q, k, my_cos, my_sin)
note('RoPE(q)', q_rope, '形状不变，只旋转')
note('RoPE(k)', k_rope, 'V 不参与')

_ref_q, _ref_k = Q3.apply_rotary_pos_emb(q, k, qwen.cos, qwen.sin)
check(q_rope, _ref_q, f'L{LAYER} RoPE(q)')
check(k_rope, _ref_k, f'L{LAYER} RoPE(k)')

✓ my_rotate_half                     max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 RoPE(q)                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 RoPE(k)                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

In [38]:
# RoPE 保长度：旋转不改变向量的模
_before = q[0, 0, -1].float().norm().item()
_after = q_rope[0, 0, -1].float().norm().item()
print(f'RoPE 前后向量模长（head 0, 最后位置）: {_before:.6f} → {_after:.6f}')
print(f'相对变化: {abs(_after - _before) / _before:.2e}   ← 旋转是保长变换')
print()
print('位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：')
_p0_diff = (q_rope[0, 0, 0] - q[0, 0, 0]).abs().max().item()
print(f'  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = {_p0_diff:.3e}')

RoPE 前后向量模长（head 0, 最后位置）: 16.679739 → 16.679739
相对变化: 0.00e+00   ← 旋转是保长变换

位置 0 的 cos=1, sin=0，所以位置 0 的 q 不应被改变：
  q_rope[0,0,0] 与 q[0,0,0] 的最大差异 = 0.000e+00


### 8.5 GQA：repeat_kv

16 个 query 头要和 8 个 kv 头对齐。做法是把每个 kv 头**复制 2 份**：

```text
k: [B, 8, S, 128]
     ↓ 插入一个长度 2 的维度并 expand
   [B, 8, 2, S, 128]
     ↓ reshape 合并前两维
   [B, 16, S, 128]
```

复制方式是 `repeat_interleave` 语义：kv 头 0 服务 q 头 0 和 1，kv 头 1 服务 q 头 2 和 3，以此类推。
`expand` 不复制内存，`reshape` 才实际展开。

这一步发生在 **RoPE 之后**。顺序很重要：如果先 repeat 再转 RoPE，就要多算一倍的旋转。

In [39]:
def my_repeat_kv(hidden_states, n_rep):
    """[B, KV, S, D] -> [B, KV*n_rep, S, D]，等价于 repeat_interleave(dim=1)。"""
    batch, n_kv, seq_len, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    expanded = hidden_states[:, :, None, :, :].expand(batch, n_kv, n_rep, seq_len, head_dim)
    return expanded.reshape(batch, n_kv * n_rep, seq_len, head_dim)


k_rep = my_repeat_kv(k_rope, N_REP)
v_rep = my_repeat_kv(v, N_REP)

note('repeat_kv(k)', k_rep, f'8 头 × {N_REP} → 16 头')
note('repeat_kv(v)', v_rep)

check(k_rep, Q3.repeat_kv(_ref_k, N_REP), f'L{LAYER} repeat_kv(k)')
check(v_rep, Q3.repeat_kv(v, N_REP), f'L{LAYER} repeat_kv(v)')

print()
print('确认复制的对应关系（q 头 i ← kv 头 i // 2）：')
for q_head in [0, 1, 2, 3, 14, 15]:
    kv_head = q_head // N_REP
    same = torch.equal(k_rep[0, q_head], k_rope[0, kv_head])
    print(f'  k_rep[head {q_head:2d}] == k_rope[head {kv_head}] : {same}')

✓ L0 repeat_kv(k)                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 repeat_kv(v)                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

确认复制的对应关系（q 头 i ← kv 头 i // 2）：
  k_rep[head  0] == k_rope[head 0] : True
  k_rep[head  1] == k_rope[head 0] : True
  k_rep[head  2] == k_rope[head 1] : True
  k_rep[head  3] == k_rope[head 1] : True
  k_rep[head 14] == k_rope[head 7] : True
  k_rep[head 15] == k_rope[head 7] : True


### 8.6 QKᵀ、mask、softmax

$$\text{scores} = \frac{Q K^\top}{\sqrt{d_k}}, \qquad d_k = 128,\ \frac{1}{\sqrt{128}} \approx 0.088388$$

$$A = \mathrm{softmax}(\text{scores} + \text{mask})$$

严格按源码 `eager_attention_forward`：

1. 缩放在 matmul **之后**乘（`torch.matmul(q, k.T) * scaling`），不是先缩放 q；
2. mask 是**加**上去的，不是乘或填充；
3. softmax 指定 `dtype=torch.float32`，算完再转回 q 的 dtype。

shape 变化是整个 Attention 里最需要盯住的一段：
`[B, 16, S, 128] @ [B, 16, 128, S] → [B, 16, S, S]`。序列维度出现了两次，
第一个 S 是"谁在看"，第二个 S 是"看谁"。

In [40]:
scores = torch.matmul(q_rope, k_rep.transpose(2, 3)) * SCALING
note('QKᵀ * scaling', scores, '[B,16,S,128] @ [B,16,128,S]')

scores_masked = scores + my_mask
note('+ causal mask', scores_masked, '加性 mask')

attn_weights = torch.softmax(scores_masked, dim=-1, dtype=torch.float32).to(q_rope.dtype)
note('softmax(dim=-1)', attn_weights, '每行和为 1')

check(attn_weights, L0.attn_weights, f'L{LAYER} attn_weights')

print()
print(f'每行和是否为 1: {torch.allclose(attn_weights.sum(-1), torch.ones(1, N_HEADS, SEQ))}')
print(f'被 mask 的位置权重最大值: {attn_weights[0, 0, 0, 1:].max().item():.3e}   ← 应为 0')

✓ L0 attn_weights                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00

每行和是否为 1: True
被 mask 的位置权重最大值: 0.000e+00   ← 应为 0


In [41]:
# 观察窗口：head 0 的完整 9×9 attention 矩阵。
# look 认得 [B, heads, S, S] 这个形状，直接画成 token × token 网格。
look(attn_weights, f'Layer {LAYER}, head 0')

Layer 0, head 0  (行=query 位置，列=key 位置，值 ×100)
               北京 是中国     的   首都     ，   巴黎     是   法国     的
      北京 │  100.0      ·      ·      ·      ·      ·      ·      ·      ·
    是中国 │   44.7   55.3      ·      ·      ·      ·      ·      ·      ·
        的 │    3.8   68.6   27.5      ·      ·      ·      ·      ·      ·
      首都 │   39.0    3.8   13.4   43.8      ·      ·      ·      ·      ·
        ， │    0.7    5.6   13.4    0.2   80.1      ·      ·      ·      ·
      巴黎 │    0.7    2.6   41.6    8.8   37.1    9.2      ·      ·      ·
        是 │    0.4    1.4   35.5    0.2   49.7    1.5   11.3      ·      ·
      法国 │    2.2    2.5   10.3    5.1   20.6   42.7   12.3    4.4      ·
        的 │    1.6   10.6   21.2    0.6   22.5    0.5   16.9    0.4   25.8


### 8.7 加权求和 V，然后 o_proj

$$\text{output} = A V$$

`[B, 16, S, S] @ [B, 16, S, 128] → [B, 16, S, 128]`。第二个 S 被消掉了。

之后要把 16 个头拼回一条向量再过 `o_proj`：

```text
[B, 16, S, 128]
    ↓ transpose(1, 2)      把 S 换回第 1 维
[B, S, 16, 128]
    ↓ contiguous().reshape  16×128 = 2048 拼平
[B, S, 2048]
    ↓ o_proj                2048 → 1024
[B, S, 1024]
```

`transpose` 之后必须 `contiguous()` 才能 `reshape`，因为 transpose 只改了 stride 没搬内存。
这一步的顺序不能反：先 transpose 再 reshape，才能保证同一个位置的 16 个头被拼在一起。

In [42]:
attn_out_heads = torch.matmul(attn_weights, v_rep)
note('A @ V', attn_out_heads, '消掉 key 维')

attn_out_merged = attn_out_heads.transpose(1, 2).contiguous().reshape(BATCH, SEQ, -1)
note('transpose + reshape', attn_out_merged, '16 头拼成 2048')

attn_output = my_linear(attn_out_merged, P0.o_proj)
note('o_proj', attn_output, '2048 → 1024')

check(attn_output, L0.o_proj, f'L{LAYER} o_proj')
# Attention 模块的最终输出就是 o_proj 的输出，工具已经把 tuple 拆开了
check(attn_output, L0.attn_out, f'L{LAYER} self_attn 输出')

✓ L0 o_proj                          max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 self_attn 输出                    max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 8.8 第一个残差连接

$$h = x + \mathrm{Attention}(\mathrm{RMSNorm}(x))$$

加的是 §8.1 记下来的 `residual_1`，也就是**未经归一化**的 layer 输入。

In [43]:
hidden_after_attn = residual_1 + attn_output
note('residual add ①', hidden_after_attn, '加未归一化的 layer 输入')

residual_2 = hidden_after_attn

print('残差前后的量级对比（最后位置，L2 范数）：')
print(f'  residual (layer 输入)  {residual_1[0, -1].norm().item():9.4f}')
print(f'  attention 输出         {attn_output[0, -1].norm().item():9.4f}')
print(f'  相加之后               {hidden_after_attn[0, -1].norm().item():9.4f}')
print()
look(hidden_after_attn, 'after residual ①')
# 工具在 hook 里也拼了同一个中间态，顺带确认两边一致
check(hidden_after_attn, L0.after_attn, f'L{LAYER} residual add ①')

残差前后的量级对比（最后位置，L2 范数）：
  residual (layer 输入)     0.8380
  attention 输出            5.5856
  相加之后                  5.5742

after residual ①            (1, 9, 1024)          
                             [0, -1, :6] = [-0.6333, -0.3618, -0.0587, -0.1963, +0.1306, +0.1057]
✓ L0 residual add ①                  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


True

### 8.9 MLP：SwiGLU

$$\mathrm{MLP}(x) = W_{\text{down}} \big( \mathrm{SiLU}(W_{\text{gate}} x) \odot W_{\text{up}} x \big)$$

三个投影，两条并行的路径：

```text
        x  [B, S, 1024]
        ├──── gate_proj ───→ [B, S, 3072] ──→ SiLU ──┐
        │                                            ⊙  逐元素相乘
        └──── up_proj ─────→ [B, S, 3072] ───────────┘
                                    │
                              down_proj
                                    ▼
                             [B, S, 1024]
```

`gate` 和 `up` 是两个**独立的**权重矩阵，不是同一个矩阵切两半。
激活只作用在 gate 分支上，up 分支保持线性。

In [44]:
h2 = my_rmsnorm(hidden_after_attn, P0.norm2)
note('post_attention_layernorm', h2)
check(h2, L0.norm2, f'L{LAYER} post_attn_norm')

gate = my_linear(h2, P0.gate)
up = my_linear(h2, P0.up)
note('gate_proj', gate, '1024 → 3072')
note('up_proj', up, '1024 → 3072')
check(gate, L0.gate, f'L{LAYER} gate_proj')
check(up, L0.up, f'L{LAYER} up_proj')

activated = my_silu(gate)
gated = activated * up
note('SiLU(gate)', activated)
note('SiLU(gate) * up', gated, '逐元素相乘')

mlp_output = my_linear(gated, P0.down)
note('down_proj', mlp_output, '3072 → 1024')
check(mlp_output, L0.down, f'L{LAYER} down_proj')
check(mlp_output, L0.mlp_out, f'L{LAYER} mlp 输出')

✓ L0 post_attn_norm                  max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00


✓ L0 gate_proj                       max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 up_proj                         max_abs=0.000e+00  mean_abs=0.000e+00  max_rel=0.000e+00
✓ L0 down_proj                       max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08
✓ L0 mlp 输出                          max_abs=2.384e-07  mean_abs=1.458e-08  max_rel=7.242e-08


True

In [45]:
print('gate 分支与 up 分支的数值分布（最后位置，3072 维）：')
for name, tensor in [('gate（激活前）', gate), ('SiLU(gate)', activated),
                     ('up', up), ('相乘之后', gated)]:
    values = tensor[0, -1].float()
    print(f'  {name:<14s} min={values.min():+8.3f}  max={values.max():+8.3f}  '
          f'mean={values.mean():+7.4f}  std={values.std():6.4f}')
print()
_negative_ratio = (gate[0, -1] < 0).float().mean().item()
print(f'gate 为负的比例: {_negative_ratio:.1%}  '
      f'← SiLU 把负值压向 0，起到门控作用')

gate 分支与 up 分支的数值分布（最后位置，3072 维）：


  gate（激活前）      min=  -4.120  max=  +2.457  mean=-0.5787  std=0.7180
  SiLU(gate)     min=  -0.278  max=  +2.263  mean=-0.1244  std=0.1570
  up             min=  -2.989  max=  +1.729  mean=-0.0079  std=0.2873
  相乘之后           min=  -3.699  max=  +0.743  mean=-0.0003  std=0.0962

gate 为负的比例: 83.5%  ← SiLU 把负值压向 0，起到门控作用


### 8.10 第二个残差连接，Layer 0 完成

In [46]:
layer_output = residual_2 + mlp_output
note('residual add ②', layer_output)
note('layer output', layer_output, '= 下一层的 inp')

check(layer_output, L0.out, f'L{LAYER} layer 输出')
# 下一层的 inp 就是这一层的 out，顺带确认这条链是连着的
check(layer_output, qwen.L[LAYER + 1].inp, f'L{LAYER} 输出 == L{LAYER + 1} 输入')

✓ L0 layer 输出                        max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08
✓ L0 输出 == L1 输入                     max_abs=2.384e-07  mean_abs=1.465e-08  max_rel=3.891e-08


True

In [47]:
trace()

shape 变化记录（38 个节点）
──────────────────────────────────────────────────────────────────────
input_ids                 (1, 9)   # B=1, S=9 的整数张量
embed_tokens              (1, 9, 1024)   # 查表：行号 → 1024 维向量
layers.0                  (1, 9, 1024)   # 保形
layers.1                  (1, 9, 1024)   # 保形  ⋯ 中间 26 层省略
layers.27                 (1, 9, 1024)   # 保形
final_norm                (1, 9, 1024)   # RMSNorm，不改形状
lm_head                   (1, 9, 151936)   # 1024 → 151936 词表打分
layer input               (1, 9, 1024)   # = embedding 输出
input_layernorm           (1, 9, 1024)   # RMSNorm，形状不变
q_proj                    (1, 9, 2048)   # 16 头 × 128
k_proj                    (1, 9, 1024)   # 8 头 × 128
v_proj                    (1, 9, 1024)   # 8 头 × 128
q view                    (1, 9, 16, 128)   # 2048 拆成 16×128
k view                    (1, 9, 8, 128)   # 1024 拆成 8×128
q_norm                    (1, 9, 16, 128)   # 在 head_dim=128 上归一化
k_norm                    (1, 9, 8, 128)   # V 不做 norm
q transpose(

上面是 `note()` 一路登记下来的全部形状节点：前 7 行来自 §4 的顶层骨架，其余是 Layer 0 的内部。
凡是有官方对照物的节点，前面都已经 `check` 过了。

形状变化可以归成三段：

- **升维**：1024 → 2048（q_proj）或 1024 → 3072（gate/up_proj）；
- **序列维出现两次**：`[B, 16, S, S]` 是唯一一处形状与序列长度成平方关系的张量；
- **降回 1024**：o_proj 和 down_proj 把宽度还原，残差才能相加。

整个 Layer 是保形的：进去 `[1, 9, 1024]`，出来 `[1, 9, 1024]`。

## 9. 封装成模块，跑完 28 层

把上面验证过的函数组装成与 Qwen3 源码结构对应的类：

```text
MyQwen3ForCausalLM
└── MyQwen3Model
    ├── my_embedding
    ├── MyQwen3DecoderLayer × 28
    │   ├── MyQwen3RMSNorm      (input_layernorm)
    │   ├── MyQwen3Attention
    │   │   ├── MyQwen3RMSNorm  (q_norm / k_norm)
    │   │   └── RoPE / GQA / causal attention
    │   ├── MyQwen3RMSNorm      (post_attention_layernorm)
    │   └── MyQwen3MLP
    ├── MyQwen3RMSNorm          (final norm)
    └── lm_head（与 embedding 共享权重）
```

参数全部从 §3.1 的目录 `W` 取（`W.L[i].q_proj` 等），拿到的就是真实模型的张量引用，**不复制、不重新初始化**。

每个类的构造参数从"一个源模块"改成"一个层号"，因为 `W.L[i]` 已经把那一层的 11 个权重摆好了。

In [48]:
class MyQwen3RMSNorm:
    """对应 Qwen3RMSNorm。"""

    def __init__(self, weight, eps=RMS_EPS):
        self.weight = weight
        self.eps = eps

    def __call__(self, x):
        return my_rmsnorm(x, self.weight, self.eps)


class MyQwen3MLP:
    """对应 Qwen3MLP，SwiGLU 结构。权重取自 W.L[index]。"""

    def __init__(self, index):
        p = W.L[index]
        self.gate_weight = p.gate
        self.up_weight = p.up
        self.down_weight = p.down

    def __call__(self, x):
        gate = my_linear(x, self.gate_weight)
        up = my_linear(x, self.up_weight)
        return my_linear(my_silu(gate) * up, self.down_weight)

In [49]:
class MyQwen3Attention:
    """对应 Qwen3Attention + eager_attention_forward。"""

    def __init__(self, index):
        p = W.L[index]
        self.q_weight = p.q_proj
        self.k_weight = p.k_proj
        self.v_weight = p.v_proj
        self.o_weight = p.o_proj
        self.q_norm = MyQwen3RMSNorm(p.q_norm)
        self.k_norm = MyQwen3RMSNorm(p.k_norm)

    def __call__(self, x, cos, sin, mask, collect=None):
        batch, seq_len, _ = x.shape
        head_shape = (batch, seq_len, -1, HEAD_DIM)

        # 投影 → 拆头 → 逐头 norm → 头维提前
        q = self.q_norm(my_linear(x, self.q_weight).view(head_shape)).transpose(1, 2)
        k = self.k_norm(my_linear(x, self.k_weight).view(head_shape)).transpose(1, 2)
        v = my_linear(x, self.v_weight).view(head_shape).transpose(1, 2)

        q, k = my_apply_rope(q, k, cos, sin)          # RoPE 在 norm 之后
        k = my_repeat_kv(k, N_REP)                    # GQA 在 RoPE 之后
        v = my_repeat_kv(v, N_REP)

        scores = torch.matmul(q, k.transpose(2, 3)) * SCALING
        if mask is not None:
            scores = scores + mask
        weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q.dtype)

        out = torch.matmul(weights, v).transpose(1, 2).contiguous()
        out = out.reshape(batch, seq_len, -1)
        if collect is not None:
            collect['attn_weights'] = weights
        return my_linear(out, self.o_weight)

In [50]:
class MyQwen3DecoderLayer:
    """对应 Qwen3DecoderLayer。pre-norm + 两次残差。"""

    def __init__(self, index):
        p = W.L[index]
        self.input_layernorm = MyQwen3RMSNorm(p.norm1)
        self.self_attn = MyQwen3Attention(index)
        self.post_attention_layernorm = MyQwen3RMSNorm(p.norm2)
        self.mlp = MyQwen3MLP(index)

    def __call__(self, hidden_states, cos, sin, mask, collect=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, cos, sin, mask, collect)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        return residual + hidden_states


class MyQwen3Model:
    """对应 Qwen3Model：embedding + 28 层 + final norm。"""

    def __init__(self):
        self.embed_weight = W.embed
        self.layers = [MyQwen3DecoderLayer(i) for i in range(N_LAYERS)]
        self.norm = MyQwen3RMSNorm(W.final_norm)

    def __call__(self, ids, collect_attn=False):
        seq_len = ids.shape[1]
        position_ids = torch.arange(seq_len, device=ids.device).unsqueeze(0)
        cos, sin, _ = my_rope_tables(position_ids)
        mask = my_causal_mask(seq_len)

        hidden = my_embedding(ids, self.embed_weight)
        all_hidden = [hidden]
        all_attn = []
        for layer in self.layers:
            bucket = {} if collect_attn else None
            hidden = layer(hidden, cos, sin, mask, bucket)
            all_hidden.append(hidden)
            if collect_attn:
                all_attn.append(bucket['attn_weights'])
        return self.norm(hidden), all_hidden, all_attn

In [51]:
class MyQwen3ForCausalLM:
    """对应 Qwen3ForCausalLM。lm_head 与 embedding 共享权重。"""

    def __init__(self):
        self.model = MyQwen3Model()
        # 权重绑定：lm_head 与 embedding 是同一个张量，直接引用，不复制
        self.lm_head_weight = W.embed

    def __call__(self, ids, collect_attn=False):
        last_hidden, all_hidden, all_attn = self.model(ids, collect_attn)
        logits = my_linear(last_hidden, self.lm_head_weight)
        return logits, last_hidden, all_hidden, all_attn


my_model = MyQwen3ForCausalLM()
print(f'复现模型层数: {len(my_model.model.layers)}')
print(f'lm_head 权重与 embedding 是同一对象: '
      f'{my_model.lm_head_weight is my_model.model.embed_weight}')
print(f'与官方模型共享权重（未复制）: '
      f'{my_model.model.layers[0].mlp.gate_weight is model.model.layers[0].mlp.gate_proj.weight}')

复现模型层数: 28
lm_head 权重与 embedding 是同一对象: True
与官方模型共享权重（未复制）: True


### 9.1 跑完整个模型，逐层验证

关键设计：**每层都用自己上一层的输出作为输入**，不从官方结果里"借"中间态。
这样误差会累积，才能真正检验复现的正确性。如果每层都拿 `qwen.L[i].inp` 当输入，
每层误差都会被重置，验证的强度大打折扣。

在比对之前必须先确认一件事：`output_hidden_states` 返回的 29 个张量，最后一个到底是什么。

第一次写这段验证时我假设 `hidden_states[28]` 就是 Layer 27 的输出，结果那一层报出 460 的巨大误差，
而 attention 权重完全正确——这种"只有最后一层错、且错得离谱"的形态说明不是数值累积，是语义理解错了。
下面用 hook 直接查证。

In [52]:
# 这是全本唯一一处直接读 official.hidden_states 的地方，因为要查证的正是它的下标语义。
_l27_hook = qwen.L[N_LAYERS - 1].out          # hook 抓的 Layer 27 原始输出
_norm_hook = qwen.final_norm                  # hook 抓的 final RMSNorm 输出
_hs28 = qwen.official.hidden_states[N_LAYERS]

print(f'hidden_states[{N_LAYERS}] == Layer 27 的 hook 输出 : '
      f'{torch.equal(_hs28, _l27_hook)}')
print(f'hidden_states[{N_LAYERS}] == final_norm 的输出    : '
      f'{torch.equal(_hs28, _norm_hook)}')
print()
print(f'L2 范数对比:')
print(f'  Layer 27 输出（norm 前） {_l27_hook.norm().item():10.3f}')
print(f'  final_norm 输出          {_norm_hook.norm().item():10.3f}')
print(f'  hidden_states[28]        {_hs28.norm().item():10.3f}')
print()
print('这就是为什么工具让 qwen.L[i].out 一律取自 hook：')
print('后面 28 层的验证只要写 qwen.L[i].out，下标语义的坑在代码层面就无从踩到。')

hidden_states[28] == Layer 27 的 hook 输出 : False
hidden_states[28] == final_norm 的输出    : True

L2 范数对比:
  Layer 27 输出（norm 前）   1510.185
  final_norm 输出             391.355
  hidden_states[28]           391.355

这就是为什么工具让 qwen.L[i].out 一律取自 hook：
后面 28 层的验证只要写 qwen.L[i].out，下标语义的坑在代码层面就无从踩到。


**`hidden_states` 的下标语义不是均匀的：**

| 下标 | 含义 |
|---|---|
| `hidden_states[0]` | embedding 输出（= Layer 0 的输入） |
| `hidden_states[i]`，1 ≤ i ≤ 27 | Layer i-1 的输出（= Layer i 的输入） |
| `hidden_states[28]` | **final RMSNorm 之后**的结果，不是 Layer 27 的原始输出 |

所以验证 Layer 27 时要拿 hook 抓到的输出比，而不是 `hidden_states[28]`。

In [53]:
my_logits, my_last_hidden, my_all_hidden, my_all_attn = my_model(
    input_ids, collect_attn=True)

print(f'my_logits        {tuple(my_logits.shape)}')
print(f'my_all_hidden    {len(my_all_hidden)} 个 × {tuple(my_all_hidden[0].shape)}')
print(f'my_all_attn      {len(my_all_attn)} 个 × {tuple(my_all_attn[0].shape)}')

my_logits        (1, 9, 151936)
my_all_hidden    29 个 × (1, 9, 1024)
my_all_attn      28 个 × (1, 16, 9, 9)


判定标准也需要说明。hidden state 里存在量级 6000 以上的元素（§9.2 会追查这件事），
对这种张量用固定的绝对公差没有意义。这里用**尺度相对误差**：

$$\text{rel} = \frac{\max |{\rm mine} - {\rm ref}|}{\max |{\rm ref}|}$$

float32 的机器精度是 `1.19e-07`。经过 28 层累积，达到 1e-6 量级属于正常范围，
判定阈值取 `1e-5`。

In [54]:
# 逐层对齐：hidden state 与 attention 权重。
# 基准一律用 qwen.L[i].out（来自 hook），不需要再操心 hidden_states 的下标。
print(f'{"层":>3s}  {"hidden max_abs":>14s}  {"hidden mean_abs":>15s}  '
      f'{"scale_rel":>11s}  {"attn max_abs":>13s}  ok')
print('─' * 74)

layer_errors = []
first_failure = None

for layer_index in range(N_LAYERS):
    mine = my_all_hidden[layer_index + 1]
    ref = qwen.L[layer_index].out
    diff = (mine.float() - ref.float()).abs()
    max_abs = diff.max().item()
    mean_abs = diff.mean().item()
    scale_rel = max_abs / ref.float().abs().max().item()

    attn_diff = (my_all_attn[layer_index].float()
                 - qwen.L[layer_index].attn_weights.float()).abs().max().item()

    ok = scale_rel < REL_THRESHOLD and attn_diff < REL_THRESHOLD
    if not ok and first_failure is None:
        first_failure = layer_index
    layer_errors.append((layer_index, max_abs, mean_abs, scale_rel, attn_diff, ok))

    flag = '✓' if ok else '✗'
    print(f'{layer_index:>3d}  {max_abs:>14.3e}  {mean_abs:>15.3e}  '
          f'{scale_rel:>11.3e}  {attn_diff:>13.3e}  {flag}')

print('─' * 74)
_all_ok = all(row[5] for row in layer_errors)
print(f'全部 {N_LAYERS} 层通过（scale_rel < {REL_THRESHOLD:.0e}）: {_all_ok}')
if first_failure is not None:
    print(f'⚠ 首个不一致的层: Layer {first_failure}')
else:
    print('未出现不一致，无需定位首个误差节点。')

record(f'全部 {N_LAYERS} 层 hidden state', _all_ok,
       max(row[3] for row in layer_errors))
record(f'全部 {N_LAYERS} 层 attention 权重',
       all(row[4] < REL_THRESHOLD for row in layer_errors),
       max(row[4] for row in layer_errors))

  层  hidden max_abs  hidden mean_abs    scale_rel   attn max_abs  ok
──────────────────────────────────────────────────────────────────────────
  0       2.384e-07        1.465e-08    3.891e-08      0.000e+00  ✓
  1       1.192e-06        6.816e-08    1.344e-07      8.643e-07  ✓
  2       1.465e-03        3.720e-07    2.291e-07      1.252e-06  ✓
  3       1.465e-03        4.077e-07    2.292e-07      7.153e-07  ✓
  4       1.465e-03        4.383e-07    2.292e-07      8.941e-07  ✓
  5       1.465e-03        4.893e-07    2.292e-07      7.153e-07  ✓
  6       1.465e-03        5.390e-07    2.293e-07      1.341e-06  ✓
  7       1.465e-03        5.955e-07    2.293e-07      9.537e-07  ✓
  8       1.465e-03        6.520e-07    2.295e-07      1.192e-06  ✓
  9       1.465e-03        7.016e-07    2.297e-07      1.192e-06  ✓
 10       1.465e-03        7.973e-07    2.297e-07      9.835e-07  ✓
 11       1.465e-03        8.944e-07    2.298e-07      1.073e-06  ✓
 12       1.465e-03        9.582e-07    

True

### 9.2 那个恒定不变的 max_abs 是什么

上表里 `hidden max_abs` 从第 2 层起就锁定在 `1.465e-03` 不再变化，而 `mean_abs` 一直在缓慢增长。
最大误差不随层数增长，说明它不是累积效应，而是**某个特定元素的浮点精度极限**。

In [55]:
_probe = qwen.L[9].out          # 网络中段任取一层的输出
print(f'Layer 9 输出的绝对值分布: max={_probe.abs().max().item():.1f}  '
      f'mean={_probe.abs().mean().item():.4f}')
print()
print('最大的 5 个元素：')
_top = _probe.abs().flatten().topk(5)
for value, flat_index in zip(_top.values, _top.indices):
    seq_pos = (flat_index // HIDDEN % SEQ).item()
    channel = (flat_index % HIDDEN).item()
    print(f'  |{value.item():9.1f}|  在 [0, {seq_pos}, {channel}]')
print()
_ulp = torch.finfo(torch.float32).eps * _probe.abs().max().item()
_observed = max(row[1] for row in layer_errors)      # 28 层里最大的 max_abs
print(f'float32 在 |x|≈{_probe.abs().max().item():.0f} 处的最小间隔（1 ULP）= {_ulp:.3e}')
print(f'上表观察到的 max_abs = {_observed:.3e} ≈ {_observed / _ulp:.1f} ULP')
print()
print('结论：最大误差来自数值最大的那个元素，是 float32 表示精度的下限，不是实现错误。')

Layer 9 输出的绝对值分布: max=6378.2  mean=1.6934

最大的 5 个元素：
  |   6378.2|  在 [0, 0, 35]
  |    628.1|  在 [0, 0, 13]
  |    364.1|  在 [0, 0, 1]
  |    154.3|  在 [0, 0, 277]
  |    106.9|  在 [0, 0, 7]

float32 在 |x|≈6378 处的最小间隔（1 ULP）= 7.603e-04
上表观察到的 max_abs = 1.465e-03 ≈ 1.9 ULP

结论：最大误差来自数值最大的那个元素，是 float32 表示精度的下限，不是实现错误。


这里浮出一个意料之外的现象：**位置 0 的第 35 号通道，数值高达 6378，比全张量均值大了三个数量级。**
这不是我们预设要观察的东西，是验证过程中撞见的。放到 §11 一起看它沿层的演化。

## 10. 出口：final norm → LM Head → Top-K → token

```text
   Layer 27 输出  [B, S, 1024]
         ↓
   final RMSNorm                    model.model.norm
         ↓        [B, S, 1024]
         ↓
   lm_head        ← 复用 embedding 那张 [151936, 1024] 表
         ↓        [B, S, 151936]
         ↓
   取最后一个位置  [151936]
         ↓
   Top-K
         ↓
   token id → decode
```

LM Head 的计算就是 `hidden @ embed_weight.T`：把 1024 维的 hidden state 与词表里 151936 个
token 向量逐个做内积。**内积大 = 方向接近 = 得分高。**
入口那次查表和出口这次打分，用的是同一张矩阵。

In [56]:
my_final_hidden = my_model.model.norm(my_all_hidden[-1])

check(my_final_hidden, qwen.final_norm, 'final RMSNorm')
check(my_logits, qwen.logits, 'lm_head logits')

print()
print(f'官方 logits 数值范围: [{qwen.logits.min().item():.3f}, '
      f'{qwen.logits.max().item():.3f}]')
print(f'复现 logits 数值范围: [{my_logits.min().item():.3f}, {my_logits.max().item():.3f}]')

✓ final RMSNorm                      max_abs=8.297e-05  mean_abs=2.408e-06  max_rel=1.051e-06
✓ lm_head logits                     max_abs=3.242e-05  mean_abs=3.097e-06  max_rel=1.373e-06

官方 logits 数值范围: [-18.614, 23.609]
复现 logits 数值范围: [-18.614, 23.609]


这两项的 `max_abs` 比前面各节点大得多（logits 到了 1e-2 量级），但 `max_rel` 仍在 1e-6。
原因是 §9.2 那个量级 6000+ 的元素经过 final norm 和一次 1024→151936 的矩阵乘后，
绝对误差被同比例放大了——这正是必须用尺度相对误差、不能用固定公差的场景。
早先这里曾经手填 `atol=1e-2` 把它压过去，那是自相矛盾的做法：
判据应该只有一个，而不是每个节点单独调一次。

除了相对误差，还有一个更硬的判据：**预测本身是否一致**。下面直接验证。

In [57]:
TOP_K = 10
last_position = SEQ - 1

official_last = qwen.logits[0, last_position]
my_last = my_logits[0, last_position]

official_top = official_last.topk(TOP_K)
my_top = my_last.topk(TOP_K)

official_probs = torch.softmax(official_last, dim=-1)

print(f'输入: {PROMPT}')
print(f'预测第 {SEQ + 1} 个 token（基于位置 {last_position} 的 logits）\n')
print(f'{"排名":>4s}  {"官方 token":<12s} {"logit":>8s} {"概率":>8s}   '
      f'{"复现 token":<12s} {"logit":>8s}  一致')
print('─' * 74)
for rank in range(TOP_K):
    o_id = official_top.indices[rank].item()
    m_id = my_top.indices[rank].item()
    o_text = tokenizer.decode([o_id])
    m_text = tokenizer.decode([m_id])
    prob = official_probs[o_id].item()
    same = '✓' if o_id == m_id else '✗'
    print(f'{rank + 1:>4d}  {o_text!r:<12s} {official_top.values[rank].item():>8.3f} '
          f'{prob:>7.2%}   {m_text!r:<12s} {my_top.values[rank].item():>8.3f}  {same}')

输入: 北京是中国的首都，巴黎是法国的
预测第 10 个 token（基于位置 8 的 logits）

  排名  官方 token        logit       概率   复现 token        logit  一致
──────────────────────────────────────────────────────────────────────────
   1  '首都'           23.249  95.94%   '首都'           23.249  ✓
   2  '首'            19.600   2.50%   '首'            19.600  ✓
   3  '象征'           17.285   0.25%   '象征'           17.285  ✓
   4  '都'            17.098   0.20%   '都'            17.098  ✓
   5  '国家'           16.898   0.17%   '国家'           16.898  ✓
   6  '____'         15.765   0.05%   '____'         15.765  ✓
   7  '代表'           15.759   0.05%   '代表'           15.759  ✓
   8  '代'            15.716   0.05%   '代'            15.716  ✓
   9  '省'            15.660   0.05%   '省'            15.660  ✓
  10  '第二'           15.353   0.04%   '第二'           15.353  ✓


In [58]:
# 闭环验证：官方与复现的 argmax 预测是否一致（全部 9 个位置）
official_argmax = qwen.logits[0].argmax(-1)
my_argmax = my_logits[0].argmax(-1)
all_match = torch.equal(official_argmax, my_argmax)

print(f'{"位置":>4s}  {"输入 token":<10s} → {"官方预测":<10s} {"复现预测":<10s} 一致')
print('─' * 52)
for position in range(SEQ):
    source = tokenizer.decode([input_ids[0, position].item()])
    o_pred = tokenizer.decode([official_argmax[position].item()])
    m_pred = tokenizer.decode([my_argmax[position].item()])
    flag = '✓' if official_argmax[position] == my_argmax[position] else '✗'
    print(f'{position:>4d}  {source!r:<10s} → {o_pred!r:<10s} {m_pred!r:<10s}  {flag}')
print('─' * 52)
print(f'全部 {SEQ} 个位置预测一致: {all_match}')

record('Top-K 预测一致（全部位置）', all_match)

  位置  输入 token   → 官方预测       复现预测       一致
────────────────────────────────────────────────────
   0  '北京'       → '地铁'       '地铁'        ✓
   1  '是中国'      → '的'        '的'         ✓
   2  '的'        → '首都'       '首都'        ✓
   3  '首都'       → '，'        '，'         ✓
   4  '，'        → '也是'       '也是'        ✓
   5  '巴黎'       → '是'        '是'         ✓
   6  '是'        → '法国'       '法国'        ✓
   7  '法国'       → '的'        '的'         ✓
   8  '的'        → '首都'       '首都'        ✓
────────────────────────────────────────────────────
全部 9 个位置预测一致: True


True

每个位置都在预测"它的下一个 token"，这是 causal 语言模型的定义。位置 8（最后一个 `的`）
的预测才是我们关心的续写结果。前面几个位置的预测顺带展示了模型在读到一半时的判断。

In [59]:
# 把预测接回原文，完成一次完整闭环
next_token_id = official_argmax[last_position].item()
continuation = tokenizer.decode([next_token_id])

print(f'原文:   {PROMPT}')
print(f'续写:   {PROMPT}{continuation}')
print()
print(f'预测 token id = {next_token_id}, 文本 = {continuation!r}')
print(f'概率 = {official_probs[next_token_id].item():.2%}')
print()
_second = official_top.indices[1].item()
print(f'与第二名的 logit 差距: '
      f'{official_top.values[0].item() - official_top.values[1].item():.3f}')
print(f'第二名: {tokenizer.decode([_second])!r} '
      f'({official_probs[_second].item():.2%})')

原文:   北京是中国的首都，巴黎是法国的
续写:   北京是中国的首都，巴黎是法国的首都

预测 token id = 106114, 文本 = '首都'
概率 = 95.94%

与第二名的 logit 差距: 3.649
第二名: '首' (2.50%)


## 11. 让数据决定看哪几层

28 层结构相同，但行为不一定相同。这一节先算出每层的统计量，**再**根据数据挑代表层，
而不是预先指定"看第 0、13、27 层"。

统计量选这几个，都能从已有记录直接算：

- **hidden 范数**：这一层输出的量级；
- **相对变化**：`‖layer_out − layer_in‖ / ‖layer_in‖`，衡量这层改动了多少；
- **attention 熵**：注意力分布的集中程度，低熵表示聚焦在少数位置；
- **对角占比**：attention 权重落在"看自己"位置上的比例；
- **首位占比**：attention 权重落在位置 0 上的比例。

In [60]:
import math

stats = []
for layer_index in range(N_LAYERS):
    view = qwen.L[layer_index]
    layer_in = view.inp                                  # 这一层的输入（hook 抓的）
    layer_out = view.out                                 # 这一层的输出
    weights = view.attn_weights[0].float()               # [heads, S, S]

    norm_out = layer_out.float().norm().item()
    delta = (layer_out.float() - layer_in.float()).norm().item()
    rel_change = delta / layer_in.float().norm().item()

    # 只统计有效（未被 mask）的行，逐 query 位置算熵后取平均
    entropies = []
    for position in range(SEQ):
        row = weights[:, position, :position + 1]
        entropy = -(row * (row + 1e-12).log()).sum(-1).mean().item()
        max_entropy = math.log(position + 1) if position > 0 else 1.0
        entropies.append(entropy / max_entropy if position > 0 else 0.0)
    mean_entropy = sum(entropies[1:]) / (SEQ - 1)

    diagonal = weights.diagonal(dim1=-2, dim2=-1).mean().item()
    first_column = weights[:, 1:, 0].mean().item()
    max_activation = layer_out.abs().max().item()

    stats.append(dict(layer=layer_index, norm=norm_out, rel_change=rel_change,
                      entropy=mean_entropy, diagonal=diagonal,
                      first_col=first_column, max_act=max_activation))

print(f'{"层":>3s} {"‖out‖":>10s} {"相对变化":>9s} {"归一熵":>8s} '
      f'{"对角占比":>9s} {"首位占比":>9s} {"max|act|":>10s}')
print('─' * 66)
for row in stats:
    print(f'{row["layer"]:>3d} {row["norm"]:>10.1f} {row["rel_change"]:>9.4f} '
          f'{row["entropy"]:>8.4f} {row["diagonal"]:>9.4f} '
          f'{row["first_col"]:>9.4f} {row["max_act"]:>10.1f}')

  层      ‖out‖      相对变化      归一熵      对角占比      首位占比   max|act|
──────────────────────────────────────────────────────────────────
  0       32.7   12.3651   0.4365    0.6329    0.0622        6.1
  1       41.1    0.5952   0.6740    0.4032    0.1182        8.9
  2     6440.8  156.5395   0.5849    0.4132    0.1420     6392.5
  3     6440.2    0.0029   0.3287    0.1638    0.8101     6391.8
  4     6439.8    0.0032   0.4260    0.2252    0.7306     6391.4
  5     6439.9    0.0040   0.4225    0.2108    0.7179     6391.3
  6     6438.1    0.0040   0.4446    0.2269    0.6320     6389.5
  7     6436.4    0.0045   0.5690    0.2408    0.6409     6387.7
  8     6432.8    0.0047   0.4899    0.2165    0.6631     6383.8
  9     6427.4    0.0053   0.5019    0.2557    0.6511     6378.2
 10     6426.6    0.0065   0.6386    0.2430    0.5758     6377.0
 11     6423.6    0.0070   0.4377    0.2817    0.6154     6373.5
 12     6421.5    0.0058   0.6384    0.1982    0.5826     6371.1
 13     6419.4    0.005

### 11.1 按数据挑代表层

上表里有几处数字明显偏离邻居。用几条客观判据把它们选出来。

In [61]:
# 判据直接写死在这张表里：每行是 (说明, 指标名, 取最大还是最小, 排除的层)。
# 不需要记住指标有哪几个——列名就是上面那张表的列名。
CRITERIA_SPEC = [
    ('相对变化最大',   'rel_change', 'max', ()),
    ('输出范数最大',   'norm',       'max', ()),
    ('注意力最聚焦',   'entropy',    'min', (0,)),      # Layer 0 的熵定义上偏低，排除
    ('首位占比最高',   'first_col',  'max', ()),
    ('对角占比最高',   'diagonal',   'max', ()),
]

CRITERIA = []
for label, key, mode, exclude in CRITERIA_SPEC:
    candidates = [r for r in stats if r['layer'] not in exclude]
    chosen = (max if mode == 'max' else min)(candidates, key=lambda r: r[key])
    CRITERIA.append((label, chosen['layer'], key))
CRITERIA.append(('最后一层', N_LAYERS - 1, 'rel_change'))

print(f'{"判据":<14s} {"层":>4s}  {"该指标数值":>12s}')
print('─' * 36)
for name, layer_index, key in CRITERIA:
    print(f'{name:<14s} {layer_index:>4d}  {stats[layer_index][key]:>12.4f}')

REPRESENTATIVE = sorted({layer_index for _, layer_index, _ in CRITERIA})
print()
print(f'代表层 = {REPRESENTATIVE}')

判据                层         该指标数值
────────────────────────────────────
相对变化最大            2      156.5395
输出范数最大           25     6634.9897
注意力最聚焦           25        0.2310
首位占比最高           24        0.8940
对角占比最高            0        0.6329
最后一层             27        0.9042

代表层 = [0, 2, 24, 25, 27]


In [62]:
# 相对变化的分布：哪些层几乎什么都没做，哪些层改动巨大
_sorted = sorted(stats, key=lambda r: r['rel_change'], reverse=True)
print('相对变化排名（前 5 / 后 5）：')
for row in _sorted[:5]:
    print(f'  Layer {row["layer"]:>2d}  rel_change = {row["rel_change"]:>10.4f}')
print('  ...')
for row in _sorted[-5:]:
    print(f'  Layer {row["layer"]:>2d}  rel_change = {row["rel_change"]:>10.4f}')
print()
_middle = [r['rel_change'] for r in stats if 3 <= r['layer'] <= 25]
print(f'Layer 3–25 的相对变化: 最小 {min(_middle):.4f}  最大 {max(_middle):.4f}  '
      f'中位数 {sorted(_middle)[len(_middle) // 2]:.4f}')
print('这些层对残差流的相对改动都在百分之几以内。')

相对变化排名（前 5 / 后 5）：
  Layer  2  rel_change =   156.5395
  Layer  0  rel_change =    12.3651
  Layer 27  rel_change =     0.9042
  Layer  1  rel_change =     0.5952
  Layer 26  rel_change =     0.0898
  ...
  Layer  7  rel_change =     0.0045
  Layer  6  rel_change =     0.0040
  Layer  5  rel_change =     0.0040
  Layer  4  rel_change =     0.0032
  Layer  3  rel_change =     0.0029

Layer 3–25 的相对变化: 最小 0.0029  最大 0.0534  中位数 0.0079
这些层对残差流的相对改动都在百分之几以内。


### 11.2 追踪那个巨大的激活值

§9.2 撞见位置 0 的第 35 号通道数值达到 6378。结合 §11 的表可以看出，
`max|act|` 在 Layer 2 从 8.9 跳到 6392.5，然后一路保持到 Layer 25，最后两层又被压下去。
下面把这个通道沿 29 个 hidden state 逐层打印出来。

In [63]:
# 把整条残差流按真实顺序摊平成一个列表：embedding → 28 层输出 → final norm。
# 每个元素自带说明，不需要做下标算术，也不存在"最后一个是什么"的疑问。
STREAM = ([('embedding 输出', qwen.embed)]
          + [(f'Layer {i} 输出', qwen.L[i].out) for i in range(N_LAYERS)]
          + [('final norm 之后', qwen.final_norm)])

print(f'残差流共 {len(STREAM)} 个节点\n')
print(f'{"#":>3s}  {"节点":<18s}  {"max|act|":>10s}  {"位置":>4s}  {"通道":>5s}   |act| 均值')
print('─' * 66)
outlier_track = []
for index, (label, tensor) in enumerate(STREAM):
    flat = tensor[0].float()                                  # [S, H]
    flat_index = flat.abs().argmax().item()
    position, channel = flat_index // HIDDEN, flat_index % HIDDEN
    value = flat[position, channel].item()
    outlier_track.append((index, label, position, channel, value))
    if index <= 4 or index >= len(STREAM) - 3 or index % 6 == 0:
        print(f'{index:>3d}  {label:<18s}  {value:>10.1f}  {position:>4d}  {channel:>5d}   '
              f'{flat.abs().mean().item():>8.4f}')

_tail = outlier_track[3:]                     # 从 Layer 2 输出起
_channels = {row[3] for row in _tail}
_positions = {row[2] for row in _tail}
print()
print(f'从 Layer 2 输出起，最大激活所在通道集合 = {_channels}')
print(f'                    所在位置集合 = {_positions}')
print()
print(f'共出现 {len(_channels)} 个不同的离群通道，不是一个。分别锁定：')
print('  A: Layer 2 起长期占据榜首的那个')
print('  B: final norm 之后仍居榜首的那个')

残差流共 30 个节点

  #  节点                    max|act|    位置     通道   |act| 均值
──────────────────────────────────────────────────────────────────
  0  embedding 输出               0.1     1    297     0.0224
  1  Layer 0 输出                 6.1     4     35     0.2175
  2  Layer 1 输出                 8.9     1     35     0.2450
  3  Layer 2 输出              6392.5     0     35     1.3966
  4  Layer 3 输出              6391.8     0     35     1.4287
  6  Layer 5 输出              6391.3     0     35     1.5209
 12  Layer 11 输出             6373.5     0     35     1.8895
 18  Layer 17 输出             6362.4     0     35     2.8392
 24  Layer 23 输出             6402.2     0     35     7.0008
 27  Layer 26 输出             6097.4     0     35     8.4598
 28  Layer 27 输出              457.6     0     35     8.9885
 29  final norm 之后             78.9     2     27     2.2280

从 Layer 2 输出起，最大激活所在通道集合 = {27, 35}
                    所在位置集合 = {0, 2}

共出现 2 个不同的离群通道，不是一个。分别锁定：
  A: Layer 2 起长期占据榜首的那个
  B: final norm 

In [64]:
# A 取网络中段的榜首，B 取 final norm 之后的榜首
_, _, POS_A, CH_A, _ = outlier_track[15]        # Layer 14 输出
_, _, POS_B, CH_B, _ = outlier_track[-1]        # final norm 之后

print(f'A = 位置 {POS_A}, 通道 {CH_A}   （网络中段的榜首）')
print(f'B = 位置 {POS_B}, 通道 {CH_B}   （final norm 之后的榜首）')
print()
print(f'{"#":>3s}  {"节点":<18s}  {"A 数值":>10s}  {"B 数值":>10s}')
print('─' * 48)
for index, (label, tensor) in enumerate(STREAM):
    if index <= 3 or index >= len(STREAM) - 2 or index % 5 == 0:
        print(f'{index:>3d}  {label:<18s}  {tensor[0, POS_A, CH_A].item():>10.1f}  '
              f'{tensor[0, POS_B, CH_B].item():>10.1f}')
print()
_mid = STREAM[15][1][0]
print(f'同位置其他通道的中位量级（Layer 14 输出）: '
      f'A 处 {_mid[POS_A].abs().median().item():.4f}，'
      f'B 处 {_mid[POS_B].abs().median().item():.4f}')

A = 位置 0, 通道 35   （网络中段的榜首）
B = 位置 2, 通道 27   （final norm 之后的榜首）

  #  节点                        A 数值        B 数值
────────────────────────────────────────────────
  0  embedding 输出              -0.0         0.0
  1  Layer 0 输出                 3.1        -0.9
  2  Layer 1 输出                 1.6        -0.1
  3  Layer 2 输出              6392.5        -0.0
  5  Layer 4 输出              6391.4         0.2
 10  Layer 9 输出              6378.2        -0.0
 15  Layer 14 输出             6364.7         3.9
 20  Layer 19 输出             6375.9        23.4
 25  Layer 24 输出             6406.4        78.5
 28  Layer 27 输出              457.6       126.2
 29  final norm 之后             -2.1        78.9

同位置其他通道的中位量级（Layer 14 输出）: A 处 1.5053，B 处 0.5960


两个离群点的形态完全不同：

- **A（位置 0，通道 35）**：在 **Layer 2 内部**被一次性写入约 6390，之后 23 层几乎原样保留，
  相对改动只有百分之几。它出现在序列的第一个 token 上。
- **B（位置 2，通道 27）**：从中段开始**逐层累积**，到 Layer 26 输出时约 99。
  它是 final norm 之后仍然最大的元素。

A 解释了 §9.2 那个恒定不变的 `max_abs`：它的量级决定了 float32 在该张量上的精度下限。

结合 §11 表里"首位占比"从 Layer 3 起长期维持在 0.55–0.89，这两个现象是同一件事的两面：
大量注意力头把权重压在位置 0 上。至于机制层面的解释，超出本实验"只弄清怎么算"的范围，
留作观察记录。

### 11.3 代表层的 attention 矩阵

同一个 head 编号在不同层的行为差别很大。挑 §11.1 选出的代表层，各看一个 head。

In [65]:
for layer_index in REPRESENTATIVE:
    reasons = [name for name, index, _ in CRITERIA if index == layer_index]
    print(f'\n{"=" * 70}')
    print(f'Layer {layer_index}   入选判据: {", ".join(reasons)}')
    print(f'  归一熵={stats[layer_index]["entropy"]:.4f}  '
          f'对角={stats[layer_index]["diagonal"]:.4f}  '
          f'首位={stats[layer_index]["first_col"]:.4f}')
    look(qwen.L[layer_index].attn_weights, f'  Layer {layer_index}, head 0', head=0)


Layer 0   入选判据: 对角占比最高
  归一熵=0.4365  对角=0.6329  首位=0.0622
  Layer 0, head 0  (行=query 位置，列=key 位置，值 ×100)
               北京 是中国     的   首都     ，   巴黎     是   法国     的
      北京 │  100.0      ·      ·      ·      ·      ·      ·      ·      ·
    是中国 │   44.7   55.3      ·      ·      ·      ·      ·      ·      ·
        的 │    3.8   68.6   27.5      ·      ·      ·      ·      ·      ·
      首都 │   39.0    3.8   13.4   43.8      ·      ·      ·      ·      ·
        ， │    0.7    5.6   13.4    0.2   80.1      ·      ·      ·      ·
      巴黎 │    0.7    2.6   41.6    8.8   37.1    9.2      ·      ·      ·
        是 │    0.4    1.4   35.5    0.2   49.7    1.5   11.3      ·      ·
      法国 │    2.2    2.5   10.3    5.1   20.6   42.7   12.3    4.4      ·
        的 │    1.6   10.6   21.2    0.6   22.5    0.5   16.9    0.4   25.8

Layer 2   入选判据: 相对变化最大
  归一熵=0.5849  对角=0.4132  首位=0.1420
  Layer 2, head 0  (行=query 位置，列=key 位置，值 ×100)
               北京 是中国     的   首都     ，   巴黎     是   法国  

In [66]:
# 同一层内不同 head 的差异：用首位占比排序，看最极端的两个
LOOK_AT = REPRESENTATIVE[len(REPRESENTATIVE) // 2]
weights = qwen.L[LOOK_AT].attn_weights
per_head_first = weights[0].float()[:, 1:, 0].mean(dim=1)

print(f'Layer {LOOK_AT} 各 head 的首位占比：')
for head in range(N_HEADS):
    bar = '█' * int(per_head_first[head].item() * 40)
    print(f'  head {head:>2d}  {per_head_first[head].item():.4f}  {bar}')

_most = per_head_first.argmax().item()
_least = per_head_first.argmin().item()
print()
look(weights, f'Layer {LOOK_AT}, head {_most}（首位占比最高）', head=_most)
print()
look(weights, f'Layer {LOOK_AT}, head {_least}（首位占比最低）', head=_least)

Layer 24 各 head 的首位占比：
  head  0  0.8218  ████████████████████████████████
  head  1  0.9889  ███████████████████████████████████████
  head  2  0.5566  ██████████████████████
  head  3  0.9718  ██████████████████████████████████████
  head  4  0.8888  ███████████████████████████████████
  head  5  0.9526  ██████████████████████████████████████
  head  6  0.9389  █████████████████████████████████████
  head  7  0.9547  ██████████████████████████████████████
  head  8  0.9852  ███████████████████████████████████████
  head  9  0.6905  ███████████████████████████
  head 10  0.9660  ██████████████████████████████████████
  head 11  0.9601  ██████████████████████████████████████
  head 12  0.9403  █████████████████████████████████████
  head 13  0.9621  ██████████████████████████████████████
  head 14  0.8179  ████████████████████████████████
  head 15  0.9084  ████████████████████████████████████

Layer 24, head 1（首位占比最高）  (行=query 位置，列=key 位置，值 ×100)
               北京 是中国     的   首都     

## 12. 验证体系汇总

把散落在各节的验证结果集中列出，确认没有遗漏的节点，也确认没有"看起来通过其实没测"的项。

In [67]:
summary()

shape 变化记录（38 个节点）
──────────────────────────────────────────────────────────────────────
input_ids                 (1, 9)   # B=1, S=9 的整数张量
embed_tokens              (1, 9, 1024)   # 查表：行号 → 1024 维向量
layers.0                  (1, 9, 1024)   # 保形
layers.1                  (1, 9, 1024)   # 保形  ⋯ 中间 26 层省略
layers.27                 (1, 9, 1024)   # 保形
final_norm                (1, 9, 1024)   # RMSNorm，不改形状
lm_head                   (1, 9, 151936)   # 1024 → 151936 词表打分
layer input               (1, 9, 1024)   # = embedding 输出
input_layernorm           (1, 9, 1024)   # RMSNorm，形状不变
q_proj                    (1, 9, 2048)   # 16 头 × 128
k_proj                    (1, 9, 1024)   # 8 头 × 128
v_proj                    (1, 9, 1024)   # 8 头 × 128
q view                    (1, 9, 16, 128)   # 2048 拆成 16×128
k view                    (1, 9, 8, 128)   # 1024 拆成 8×128
q_norm                    (1, 9, 16, 128)   # 在 head_dim=128 上归一化
k_norm                    (1, 9, 8, 128)   # V 不做 norm
q transpose(

True

In [68]:
# 端到端的最终确认
_end_to_end = [
    ('logits 形状一致', tuple(my_logits.shape) == tuple(qwen.logits.shape)),
    ('argmax 预测全部一致', torch.equal(my_logits[0].argmax(-1),
                                    qwen.logits[0].argmax(-1))),
    (f'Top-{TOP_K} 排序一致', torch.equal(my_last.topk(TOP_K).indices,
                                      official_last.topk(TOP_K).indices)),
    (f'logits 相对误差 < {REL_THRESHOLD:.0e}',
     (my_logits - qwen.logits).abs().max().item()
     / qwen.logits.abs().max().item() < REL_THRESHOLD),
    ('28 层 hidden state 全部对齐', all(row[5] for row in layer_errors)),
    ('28 层 attention 权重全部对齐',
     all(row[4] < REL_THRESHOLD for row in layer_errors)),
]
for name, ok in _end_to_end:
    print(f'{"✓" if ok else "✗"} {name}')
print()
print(f'端到端闭环成立: {all(ok for _, ok in _end_to_end)}')

✓ logits 形状一致
✓ argmax 预测全部一致
✓ Top-10 排序一致
✓ logits 相对误差 < 1e-05
✓ 28 层 hidden state 全部对齐
✓ 28 层 attention 权重全部对齐

端到端闭环成立: True


## 13. 实验发现 / Experiment Findings

以下内容由实际运行数据产生，不是预先写好的结论。

### 13.1 复现结果

从 `input_ids` 到 Top-K 预测的每一个节点都与官方实现对齐。Layer 0 的 25 个节点里，
`q_proj`、`k_proj`、`v_proj`、`q_norm`、`k_norm`、RoPE、`repeat_kv`、attention 权重、
`o_proj`、`gate_proj`、`up_proj` 全部是 **max_abs = 0**，即逐位相同。
只有经过多次累加的 `down_proj` 及其下游出现 1e-7 量级差异，来自 float32 矩阵乘的求和顺序不同。

28 层各自独立向前推进（每层吃自己上一层的输出，不借用官方中间态），
尺度相对误差从 Layer 0 的 3.9e-08 增长到 Layer 27 的 1.1e-06，增长了约 27 倍，
与 28 次矩阵乘的误差累积量级相符。**9 个位置的 argmax 预测和 Top-10 排序完全一致。**

### 13.2 与预期不同的地方

**（1）`hidden_states` 的下标语义不均匀。**
最初假设 `hidden_states[28]` 是 Layer 27 的输出，验证时那一层报出 460 的误差，
而 attention 权重完全正确。这种"只有最后一层错、且错得离谱"的形态说明是语义理解错误，
不是数值问题。查证后确认 `hidden_states[28]` 已经过 final RMSNorm，
`torch.equal(hidden_states[28], final_norm 的 hook 输出)` 为 True。
这个坑的定位方式印证了 §十一 的做法有效：**逐层验证能立刻指出问题在哪一层，
而不是等到 logits 不对再回头排查。**

**（2）`rope_theta` 在 config 对象里换了位置。**
`config.json` 里它是顶层字段，但 transformers 5.15 把它移进了 `config.rope_parameters` 字典，
`hasattr(config, 'rope_theta')` 返回 False。凭记忆写 `config.rope_theta` 会直接报错。

**（3)`q_norm` / `k_norm` 的位置比预想的更靠里。**
它作用在 **reshape 之后**的 `head_dim=128` 维上，而不是 1024 维上；
`weight` 长度只有 128，被 16 个（或 8 个）头共享；V 完全没有 norm。
源码把 projection、view、norm、transpose 四个操作写在一行里，容易读漏。

### 13.3 层与层之间的差异很大

按 §11 的统计量，28 层明显分成三段：

| 层 | 相对变化 | 行为 |
|---|---|---|
| 0–2 | 12.4 / 0.60 / **156.5** | 剧烈重写。Layer 2 把残差流范数从 41 拉到 6441 |
| 3–25 | 0.003 – 0.052 | 每层只做百分之几的改动 |
| 26–27 | 0.090 / **0.904** | Layer 27 把范数从 6635 压回 1510 |

中间 23 层的相对改动中位数只有约 0.8%，但这不等于它们没用——残差流被 Layer 2 写入的
巨大常量支配，分母很大，所以相对值被压低了。

### 13.4 两个数量级异常的激活通道

验证过程中撞见的现象，不在原计划的观察清单里：

- **位置 0，通道 35**：在 Layer 2 内部被一次性写入约 **6392**，之后 23 层几乎原样保留，
  final norm 之后变成 −2.1 被彻底压掉。同位置其他通道的中位量级只有 1.5，相差三个数量级。
- **位置 2，通道 27**：从中段开始逐层累积，Layer 26 输出时约 **99**，
  且在 final norm 之后仍是全张量最大的元素（78.9）。

第一个通道直接解释了 §9.2 那个"从 Layer 2 起恒定不变的 max_abs = 1.465e-03"：
float32 在 6392 附近的最小间隔是 7.6e-04，观察到的误差只有约 2 个 ULP。
**这也说明用固定绝对公差判定这种张量是错的**，必须看尺度相对误差。

### 13.5 注意力大量集中在位置 0

"首位占比"从 Layer 3 起长期维持在 0.55–0.89。Layer 24 的 16 个 head 里，
有 11 个把 90% 以上的权重压在位置 0 上，head 1 达到 98.9%。
这与 §13.4 的通道 35 出现在位置 0 是同一现象的两面。

相比之下 Layer 0 的形态完全不同：对角占比 0.63（倾向于看自己），首位占比只有 0.06。

### 13.6 结构上确认的几件事

- **`lm_head` 与 `embed_tokens` 共享同一块内存**（`data_ptr()` 相同），
  这张 `[151936, 1024]` 的表占模型总参数的 26.1%，且不存在于 `model.safetensors` 里。
- **28 层共用同一个 `attention_mask` 对象和同一组 `(cos, sin)`**，
  RoPE 在模型级别算一次，不在 Attention 内部重复计算。
- **`q_proj` 把 1024 升到 2048**，比 hidden_size 更宽。`head_dim=128` 是配置写死的，
  不是 `hidden_size / num_heads`。
- **RoPE 是保长变换**：向量模长在旋转前后相对变化为 0；位置 0 的 cos=1、sin=0，
  该位置的 q/k 完全不变。
- **`hidden_states` 全部 29 个张量形状相同**，都是 `[1, 9, 1024]`。
- **RMSNorm 的两个参数传反不会报错。** `(1024,)` 与 `(1, 9, 1024)` 广播成功，输出形状与正确写法一模一样，结果偏差约 4%。没有任何异常可查，只能靠 `check` 发现数值不对，所以 `my_rmsnorm` 在入口断言 `weight.dim() == 1`。
- **全模型 bias 数量为 0**（现算 `named_parameters()` 里以 `.bias` 结尾的项）。Attention 的四个投影和 MLP 的三个投影都是 `bias=False`，所以 `x @ w.T` 就是完整的线性层，不缺一项。

## 14. 小结

本实验走通的完整链路：

```text
自然语言
   ↓  tokenizer（未解剖）
input_ids                 [1, 9]
   ↓  embed_tokens = 查表
hidden                    [1, 9, 1024]
   ↓  ┌─ 28 × Decoder Layer ────────────────────────┐
   ↓  │ RMSNorm → q/k/v_proj → q/k_norm → RoPE      │
   ↓  │ → repeat_kv → QKᵀ·scaling → +mask → softmax │
   ↓  │ → ×V → o_proj → ⊕residual                   │
   ↓  │ → RMSNorm → SiLU(gate)*up → down → ⊕residual│
   ↓  └─────────────────────────────────────────────┘
hidden                    [1, 9, 1024]     ← 保形
   ↓  final RMSNorm
   ↓  lm_head（复用 embedding 那张表）
logits                    [1, 9, 151936]
   ↓  取最后位置 → Top-K → decode
'首都'  (95.94%)
```

每一步都用 PyTorch 基础算子重写并与官方实现对齐，误差全部在 float32 精度范围内，
最终预测完全一致。

完成本实验后，应该能回答：

1. 一个 token 从 `input_ids` 到 logits，中间经过哪些具体的矩阵运算？
2. Q、K、V 三个投影的输出维度为什么不同，GQA 在哪一步把它们对齐？
3. `q_norm` 归一化的是哪一维，为什么是 128 而不是 1024 或 2048？
4. RoPE 的 cos/sin 在哪里计算，为什么 28 层可以共用？
5. causal mask 是怎么起作用的，为什么是加法而不是乘法？
6. 残差连接加的是归一化前还是归一化后的张量？
7. 为什么 `hidden_states` 有 29 个而不是 28 个，最后一个是什么？
8. 为什么 `model.safetensors` 里找不到 `lm_head.weight`？
9. RMSNorm 的缩放向量是 `(1024,)`、输入是 `(1, 9, 1024)`，为什么这两个参数传反了不会报错？